In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 02_vit_loss_multiseed_confirmation.ipynb
# COMPLETE SINGLE-CELL, EPOCH-RESUMABLE, RNG-FIXED VERSION
# ============================================================
#
# Main behavior
# -------------
# - Imports completed seed-42 OOF predictions from Notebook 01.
# - Trains only seeds 2025 and 2026.
# - Confirms two shortlisted ViT losses.
# - Uses the frozen five patient-level folds.
# - Uses modal_low as the derived patient reference.
# - Never loads or evaluates the official LIMUC test set.
#
# Resume behavior
# ---------------
# - Completed seed: reused.
# - Completed fold: reused.
# - Interrupted fold: resumes after the last completed epoch.
# - Interrupted epoch: repeated.
# - Model, optimizer, scheduler, AMP scaler, RNG states,
#   DataLoader generator states, patience, and history restored.
#
# Important correction
# --------------------
# Resume checkpoints are loaded on CPU. PyTorch CPU generator
# states are explicitly normalized to CPU torch.ByteTensor.
# ============================================================


# ============================================================
# 0. Install dependencies
# ============================================================

%pip -q install \
    timm==1.0.28 \
    scikit-learn==1.6.1 \
    pandas==2.2.2 \
    Pillow==11.3.0


# ============================================================
# 1. Imports
# ============================================================

import gc
import os
import json
import math
import copy
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from IPython.display import display


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 2. Development-only safety check
# ============================================================

FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "TEST_ROOT",
    "test_df",
    "test_dataset",
    "test_loader",
    "test_predictions",
    "test_patient_predictions",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    name
    for name in FORBIDDEN_TEST_OBJECTS
    if name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects exist in memory: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only this cell."
    )

print("Development-only safety check passed.")
print("Official LIMUC test data are not defined or loaded.")


# ============================================================
# 3. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 4. Paths
# ============================================================

DEVELOPMENT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
)

SCREENING_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/01_vit_loss_screening_development_only')
)

# Keep this unchanged so completed work is reused.
OUTPUT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/02_vit_loss_multiseed_confirmation_epoch_resume_v2')
)

MANIFEST_PATH = (
    SCREENING_ROOT
    / "development_image_manifest_with_folds.csv"
)

FOLD_ASSIGNMENT_PATH = (
    SCREENING_ROOT
    / "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    SCREENING_ROOT
    / "patient_reference_rule_sensitivity.csv"
)

SCREENING_ANALYSIS_PLAN_PATH = (
    SCREENING_ROOT
    / "analysis_plan_frozen_before_training.json"
)

required_paths = [
    DEVELOPMENT_ROOT,
    SCREENING_ROOT,
    MANIFEST_PATH,
    FOLD_ASSIGNMENT_PATH,
    REFERENCE_AUDIT_PATH,
    SCREENING_ANALYSIS_PLAN_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist:\n{required_path}"
        )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nDevelopment root:", DEVELOPMENT_ROOT)
print("Screening root:", SCREENING_ROOT)
print("Confirmation output:", OUTPUT_ROOT)


# ============================================================
# 5. Confirmation configuration
# ============================================================

PROTOCOL_VERSION = (
    "vit_multiseed_confirmation_epoch_resume_v2"
)

PRIMARY_REFERENCE_RULE = "modal_low"

IMPORTED_SCREENING_SEED = 42

NEW_CONFIRMATION_SEEDS = [
    2025,
    2026,
]

ALL_CONFIRMATION_SEEDS = [
    IMPORTED_SCREENING_SEED,
    *NEW_CONFIRMATION_SEEDS,
]

CANDIDATES = {
    "vit_L5_expected_ordinal_distance_a1_w020": {
        "display_name": (
            "CE + expected ordinal-distance loss, "
            "alpha=1, lambda=0.20"
        ),
        "loss_type": "ce_expected_distance",
        "ordinal_loss_weight": 0.20,
        "ordinal_alpha": 1.0,
    },

    "vit_L2_expected_value_ordinal_mae_w010": {
        "display_name": (
            "CE + expected-value ordinal MAE, "
            "lambda=0.10"
        ),
        "loss_type": "ce_expected_value_mae",
        "ordinal_loss_weight": 0.10,
        "ordinal_alpha": 1.0,
    },
}

CANDIDATE_A = (
    "vit_L5_expected_ordinal_distance_a1_w020"
)

CANDIDATE_B = (
    "vit_L2_expected_value_ordinal_mae_w010"
)

TRAINING_CONFIG = {
    "model_name": "vit_base_patch16_384",
    "pretrained": True,

    "num_classes": 4,
    "num_folds": 5,

    "image_size": 384,
    "batch_size": 8,
    "num_workers": 2,

    "epochs": 25,
    "learning_rate": 1e-4,
    "min_lr_ratio": 0.05,
    "warmup_epochs": 3,
    "weight_decay": 1e-4,

    "label_smoothing": 0.02,
    "drop_rate": 0.10,
    "drop_path_rate": 0.05,

    "early_stop_patience": 8,
    "grad_clip_norm": 1.0,

    "use_mixed_precision": True,
    "use_class_weights_in_ce": True,
    "use_weighted_sampler": False,

    "use_tta": True,
    "tta_horizontal_flip": True,

    "patient_reference": PRIMARY_REFERENCE_RULE,
    "patient_aggregation": "mean_probability",

    "epoch_checkpoint_interval": 1,

    "bootstrap_repetitions": 5000,
    "bootstrap_seed": 20260730,
}

RESUME_COMPLETED_FOLDS = True
RESUME_INTERRUPTED_FOLDS = True
SKIP_COMPLETED_SEEDS = True


# ============================================================
# 6. Atomic file-writing helpers
# ============================================================

def temporary_path(path: Path) -> Path:
    return path.with_name(
        path.name + ".temporary"
    )


def atomic_json_save(
    payload: dict,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(path)

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            default=str,
        )

    os.replace(
        temp_path,
        path,
    )


def atomic_csv_save(
    dataframe: pd.DataFrame,
    path: Path,
    index: bool = False,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(path)

    if temp_path.exists():
        temp_path.unlink()

    dataframe.to_csv(
        temp_path,
        index=index,
    )

    os.replace(
        temp_path,
        path,
    )


def atomic_torch_save(
    payload,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(path)

    if temp_path.exists():
        temp_path.unlink()

    torch.save(
        payload,
        temp_path,
    )

    os.replace(
        temp_path,
        path,
    )


def safe_torch_load(
    path: Path,
    map_location="cpu",
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


# Remove incomplete atomic temporary files.
for abandoned_file in OUTPUT_ROOT.rglob("*.temporary"):
    try:
        abandoned_file.unlink()
    except Exception:
        pass


# ============================================================
# 7. Reproducibility and RNG-state helpers
# ============================================================

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def as_cpu_byte_tensor(
    state,
    state_name: str = "RNG state",
) -> torch.Tensor:
    """
    Convert a stored generator/RNG state to the CPU uint8 tensor
    required by torch.Generator.set_state() and
    torch.set_rng_state().
    """

    if state is None:
        raise ValueError(
            f"{state_name} is missing."
        )

    if isinstance(state, torch.Tensor):
        return (
            state
            .detach()
            .to(
                device="cpu",
                dtype=torch.uint8,
            )
            .contiguous()
        )

    if isinstance(state, np.ndarray):
        return (
            torch.from_numpy(
                state.astype(
                    np.uint8,
                    copy=False,
                )
            )
            .cpu()
            .contiguous()
        )

    try:
        return torch.tensor(
            state,
            dtype=torch.uint8,
            device="cpu",
        ).contiguous()

    except Exception as error:
        raise TypeError(
            f"{state_name} could not be converted to "
            "a CPU torch.ByteTensor."
        ) from error


def capture_rng_state() -> dict:
    state = {
        "python": random.getstate(),

        "numpy": np.random.get_state(),

        "torch_cpu": (
            torch.get_rng_state()
            .cpu()
            .contiguous()
        ),

        "torch_cuda": None,
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = [
            cuda_state
            .cpu()
            .contiguous()

            for cuda_state in (
                torch.cuda.get_rng_state_all()
            )
        ]

    return state


def restore_rng_state(
    state: dict,
) -> None:
    if not isinstance(state, dict):
        raise TypeError(
            "Saved RNG state must be a dictionary."
        )

    random.setstate(
        state["python"]
    )

    numpy_state = state["numpy"]

    if isinstance(numpy_state, list):
        numpy_state = tuple(
            numpy_state
        )

    np.random.set_state(
        numpy_state
    )

    torch.set_rng_state(
        as_cpu_byte_tensor(
            state["torch_cpu"],
            state_name=(
                "PyTorch CPU RNG state"
            ),
        )
    )

    saved_cuda_states = state.get(
        "torch_cuda"
    )

    if (
        torch.cuda.is_available()
        and
        saved_cuda_states is not None
    ):
        if isinstance(
            saved_cuda_states,
            torch.Tensor,
        ):
            saved_cuda_states = [
                saved_cuda_states
            ]

        normalized_cuda_states = [
            as_cpu_byte_tensor(
                cuda_state,
                state_name=(
                    f"CUDA RNG state {state_index}"
                ),
            )

            for state_index, cuda_state in enumerate(
                saved_cuda_states
            )
        ]

        current_gpu_count = (
            torch.cuda.device_count()
        )

        if (
            len(normalized_cuda_states)
            !=
            current_gpu_count
        ):
            if (
                current_gpu_count == 1
                and
                len(normalized_cuda_states) >= 1
            ):
                normalized_cuda_states = [
                    normalized_cuda_states[0]
                ]

            else:
                raise RuntimeError(
                    "Saved CUDA RNG-state count does not "
                    "match the current GPU count.\n"
                    f"Saved: {len(normalized_cuda_states)}\n"
                    f"Current: {current_gpu_count}"
                )

        torch.cuda.set_rng_state_all(
            normalized_cuda_states
        )


def optimizer_to_device(
    optimizer,
    target_device,
) -> None:
    """
    Move loaded optimizer-state tensors to the model device.
    """

    for optimizer_state in optimizer.state.values():
        for key, value in optimizer_state.items():
            if isinstance(value, torch.Tensor):
                optimizer_state[key] = value.to(
                    target_device
                )


seed_everything(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


# ============================================================
# 8. Hash and environment helpers
# ============================================================

def file_sha256(path: Path) -> str:
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(block)

    return hasher.hexdigest()


def dictionary_sha256(dictionary: dict) -> str:
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def safe_package_version(
    package_name: str,
):
    try:
        return version(package_name)

    except PackageNotFoundError:
        return None


# ============================================================
# 9. Validate seed-42 source files
# ============================================================

screening_source_files = {}

for candidate_name, candidate_definition in (
    CANDIDATES.items()
):
    candidate_seed_directory = (
        SCREENING_ROOT
        / candidate_name
        / "seed_42"
    )

    screening_oof_path = (
        candidate_seed_directory
        / "oof_image_predictions.csv"
    )

    screening_config_path = (
        candidate_seed_directory
        / "config_used.json"
    )

    if not screening_oof_path.exists():
        raise FileNotFoundError(
            "Seed-42 OOF predictions are missing:\n"
            f"{screening_oof_path}"
        )

    if not screening_config_path.exists():
        raise FileNotFoundError(
            "Seed-42 configuration is missing:\n"
            f"{screening_config_path}"
        )

    with open(
        screening_config_path,
        "r",
        encoding="utf-8",
    ) as file:
        screening_config = json.load(file)

    exact_checks = {
        "model_name": TRAINING_CONFIG["model_name"],
        "num_classes": TRAINING_CONFIG["num_classes"],
        "num_folds": TRAINING_CONFIG["num_folds"],
        "image_size": TRAINING_CONFIG["image_size"],
        "batch_size": TRAINING_CONFIG["batch_size"],
        "epochs": TRAINING_CONFIG["epochs"],
        "warmup_epochs": TRAINING_CONFIG["warmup_epochs"],
        "use_mixed_precision": (
            TRAINING_CONFIG[
                "use_mixed_precision"
            ]
        ),
        "use_weighted_sampler": (
            TRAINING_CONFIG[
                "use_weighted_sampler"
            ]
        ),
        "use_class_weights_in_ce": (
            TRAINING_CONFIG[
                "use_class_weights_in_ce"
            ]
        ),
        "use_tta": TRAINING_CONFIG["use_tta"],
        "tta_horizontal_flip": (
            TRAINING_CONFIG[
                "tta_horizontal_flip"
            ]
        ),
        "loss_type": (
            candidate_definition[
                "loss_type"
            ]
        ),
    }

    for key, expected_value in exact_checks.items():
        observed_value = screening_config.get(key)

        if observed_value != expected_value:
            raise RuntimeError(
                f"Seed-42 configuration mismatch for "
                f"{candidate_name}: {key}\n"
                f"Expected: {expected_value}\n"
                f"Observed: {observed_value}"
            )

    numerical_checks = {
        "lr": TRAINING_CONFIG[
            "learning_rate"
        ],
        "min_lr_ratio": TRAINING_CONFIG[
            "min_lr_ratio"
        ],
        "weight_decay": TRAINING_CONFIG[
            "weight_decay"
        ],
        "label_smoothing": TRAINING_CONFIG[
            "label_smoothing"
        ],
        "drop_rate": TRAINING_CONFIG[
            "drop_rate"
        ],
        "drop_path_rate": TRAINING_CONFIG[
            "drop_path_rate"
        ],
        "grad_clip_norm": TRAINING_CONFIG[
            "grad_clip_norm"
        ],
        "ordinal_loss_weight": (
            candidate_definition[
                "ordinal_loss_weight"
            ]
        ),
        "ordinal_alpha": (
            candidate_definition[
                "ordinal_alpha"
            ]
        ),
    }

    for key, expected_value in (
        numerical_checks.items()
    ):
        observed_value = screening_config.get(key)

        if (
            observed_value is None
            or not np.isclose(
                float(observed_value),
                float(expected_value),
            )
        ):
            raise RuntimeError(
                f"Seed-42 numerical configuration mismatch "
                f"for {candidate_name}: {key}\n"
                f"Expected: {expected_value}\n"
                f"Observed: {observed_value}"
            )

    screening_source_files[
        candidate_name
    ] = {
        "oof_path": str(
            screening_oof_path
        ),

        "oof_sha256": file_sha256(
            screening_oof_path
        ),

        "config_path": str(
            screening_config_path
        ),

        "config_sha256": file_sha256(
            screening_config_path
        ),
    }

print("\nSeed-42 source configurations validated.")


# ============================================================
# 10. Load or create the frozen confirmation plan
# ============================================================

confirmation_plan_path = (
    OUTPUT_ROOT
    / "analysis_plan_frozen_before_confirmation.json"
)

if confirmation_plan_path.exists():
    with open(
        confirmation_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        existing_confirmation_plan = (
            json.load(file)
        )

    critical_plan_checks = {
        "protocol_version": PROTOCOL_VERSION,
        "primary_patient_reference": (
            PRIMARY_REFERENCE_RULE
        ),
        "candidates": CANDIDATES,
        "training_config": TRAINING_CONFIG,
        "imported_seed": (
            IMPORTED_SCREENING_SEED
        ),
        "new_training_seeds": (
            NEW_CONFIRMATION_SEEDS
        ),
        "all_confirmation_seeds": (
            ALL_CONFIRMATION_SEEDS
        ),
    }

    for key, expected_value in (
        critical_plan_checks.items()
    ):
        observed_value = (
            existing_confirmation_plan.get(
                key
            )
        )

        if observed_value != expected_value:
            raise RuntimeError(
                "Existing confirmation plan differs from "
                f"the current protocol for field: {key}\n"
                f"Expected: {expected_value}\n"
                f"Observed: {observed_value}"
            )

    expected_hash_checks = {
        "development_manifest_sha256": (
            file_sha256(MANIFEST_PATH)
        ),
        "fold_assignment_sha256": (
            file_sha256(
                FOLD_ASSIGNMENT_PATH
            )
        ),
        "reference_audit_sha256": (
            file_sha256(
                REFERENCE_AUDIT_PATH
            )
        ),
        "screening_analysis_plan_sha256": (
            file_sha256(
                SCREENING_ANALYSIS_PLAN_PATH
            )
        ),
    }

    for key, expected_hash in (
        expected_hash_checks.items()
    ):
        observed_hash = (
            existing_confirmation_plan.get(
                key
            )
        )

        if observed_hash != expected_hash:
            raise RuntimeError(
                "A frozen source file has changed since "
                f"confirmation began: {key}"
            )

    confirmation_plan_fingerprint = (
        existing_confirmation_plan[
            "confirmation_plan_fingerprint"
        ]
    )

    print(
        "\nExisting confirmation plan validated:",
        confirmation_plan_path,
    )

else:
    immutable_confirmation_plan = {
        "protocol_version": PROTOCOL_VERSION,

        "study_stage": (
            "Development-only ViT multi-seed confirmation"
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,

        "primary_patient_reference": (
            PRIMARY_REFERENCE_RULE
        ),

        "primary_patient_reference_definition": (
            "Modal image-level MES; when multiple modal "
            "classes are tied, the lower class is selected."
        ),

        "patient_prediction_aggregation": (
            "Mean image-level class probabilities per patient"
        ),

        "candidates": CANDIDATES,
        "training_config": TRAINING_CONFIG,

        "imported_seed": int(
            IMPORTED_SCREENING_SEED
        ),

        "new_training_seeds": [
            int(seed)
            for seed in NEW_CONFIRMATION_SEEDS
        ],

        "all_confirmation_seeds": [
            int(seed)
            for seed in ALL_CONFIRMATION_SEEDS
        ],

        "development_manifest_path": str(
            MANIFEST_PATH
        ),

        "development_manifest_sha256": (
            file_sha256(MANIFEST_PATH)
        ),

        "fold_assignment_path": str(
            FOLD_ASSIGNMENT_PATH
        ),

        "fold_assignment_sha256": (
            file_sha256(
                FOLD_ASSIGNMENT_PATH
            )
        ),

        "reference_audit_path": str(
            REFERENCE_AUDIT_PATH
        ),

        "reference_audit_sha256": (
            file_sha256(
                REFERENCE_AUDIT_PATH
            )
        ),

        "screening_analysis_plan_path": str(
            SCREENING_ANALYSIS_PLAN_PATH
        ),

        "screening_analysis_plan_sha256": (
            file_sha256(
                SCREENING_ANALYSIS_PLAN_PATH
            )
        ),

        "imported_seed_42_sources": (
            screening_source_files
        ),

        "checkpoint_selection_hierarchy": [
            "Validation patient QWK: higher",
            "Validation patient MAE: lower",
            "Validation patient macro F1: higher",
            "Validation image QWK: higher",
        ],

        "candidate_ranking_hierarchy": [
            "Mean OOF patient QWK across seeds: higher",
            "Mean OOF patient MAE across seeds: lower",
            "Mean OOF patient macro F1 across seeds: higher",
            "Mean OOF image QWK across seeds: higher",
        ],

        "resume_policy": (
            "Model, optimizer, scheduler, AMP scaler, "
            "early-stopping state, RNG states, DataLoader "
            "generator states, and history are saved after "
            "every completed epoch."
        ),
    }

    confirmation_plan_fingerprint = (
        dictionary_sha256(
            immutable_confirmation_plan
        )
    )

    confirmation_plan = {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "confirmation_plan_fingerprint": (
            confirmation_plan_fingerprint
        ),

        **immutable_confirmation_plan,
    }

    atomic_json_save(
        confirmation_plan,
        confirmation_plan_path,
    )

    print(
        "\nSaved frozen confirmation plan:",
        confirmation_plan_path,
    )

print(
    "Confirmation plan fingerprint:",
    confirmation_plan_fingerprint,
)


# ============================================================
# 11. Environment metadata
# ============================================================

environment_metadata = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": platform.python_version(),
    "platform": platform.platform(),

    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,

    "numpy": np.__version__,
    "pandas": pd.__version__,

    "scikit_learn": safe_package_version(
        "scikit-learn"
    ),

    "pillow": safe_package_version(
        "Pillow"
    ),

    "cuda_available": bool(
        torch.cuda.is_available()
    ),

    "cuda_version": torch.version.cuda,

    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "confirmation_plan_fingerprint": (
        confirmation_plan_fingerprint
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

atomic_json_save(
    environment_metadata,
    OUTPUT_ROOT
    / "environment_metadata.json",
)


# ============================================================
# 12. Load development manifest
# ============================================================

trainval_df = pd.read_csv(
    MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "filename",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(trainval_df.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Development manifest is missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )

trainval_df["path"] = (
    trainval_df["relative_path"]
    .map(
        lambda relative_path: str(
            DEVELOPMENT_ROOT
            / Path(relative_path)
        )
    )
)

for integer_column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:
    trainval_df[integer_column] = (
        trainval_df[integer_column]
        .astype(int)
    )

trainval_df = (
    trainval_df
    .sort_values(
        [
            "patient_id",
            "relative_path",
        ]
    )
    .reset_index(drop=True)
)

if len(trainval_df) != 9590:
    raise RuntimeError(
        f"Expected 9,590 development images; "
        f"found {len(trainval_df)}."
    )

if (
    trainval_df["patient_id"].nunique()
    != 478
):
    raise RuntimeError(
        "Expected 478 development patients; found "
        f"{trainval_df['patient_id'].nunique()}."
    )

if trainval_df["path"].duplicated().any():
    raise RuntimeError(
        "Duplicate paths exist in the manifest."
    )

observed_folds = set(
    trainval_df["cv_fold"].unique()
)

expected_folds = set(
    range(
        TRAINING_CONFIG["num_folds"]
    )
)

if observed_folds != expected_folds:
    raise RuntimeError(
        f"Expected folds {sorted(expected_folds)}; "
        f"found {sorted(observed_folds)}."
    )


# ============================================================
# 13. Validate frozen folds and references
# ============================================================

fold_assignment_df = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

reference_audit_df = pd.read_csv(
    REFERENCE_AUDIT_PATH
)

fold_assignment_df["patient_id"] = (
    fold_assignment_df["patient_id"]
    .astype(int)
)

fold_assignment_df["patient_label"] = (
    fold_assignment_df["patient_label"]
    .astype(int)
)

fold_assignment_df["cv_fold"] = (
    fold_assignment_df["cv_fold"]
    .astype(int)
)

reference_audit_df["patient_id"] = (
    reference_audit_df["patient_id"]
    .astype(int)
)

reference_audit_df[
    PRIMARY_REFERENCE_RULE
] = (
    reference_audit_df[
        PRIMARY_REFERENCE_RULE
    ]
    .astype(int)
)

reference_check = (
    fold_assignment_df[
        [
            "patient_id",
            "patient_label",
            "cv_fold",
        ]
    ]
    .merge(
        reference_audit_df[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(reference_check) != 478:
    raise RuntimeError(
        "Reference validation did not include "
        "all 478 patients."
    )

if not (
    reference_check["patient_label"]
    ==
    reference_check[
        PRIMARY_REFERENCE_RULE
    ]
).all():
    raise RuntimeError(
        "Frozen patient labels do not match modal_low."
    )

manifest_patient_check = (
    trainval_df[
        [
            "patient_id",
            "cv_fold",
            "patient_y_true",
        ]
    ]
    .drop_duplicates()
    .merge(
        fold_assignment_df[
            [
                "patient_id",
                "cv_fold",
                "patient_label",
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_manifest",
            "_frozen",
        ),
    )
)

if len(manifest_patient_check) != 478:
    raise RuntimeError(
        "Manifest validation did not include "
        "all 478 patients."
    )

if not (
    manifest_patient_check[
        "cv_fold_manifest"
    ]
    ==
    manifest_patient_check[
        "cv_fold_frozen"
    ]
).all():
    raise RuntimeError(
        "Manifest folds differ from frozen folds."
    )

if not (
    manifest_patient_check[
        "patient_y_true"
    ]
    ==
    manifest_patient_check[
        "patient_label"
    ]
).all():
    raise RuntimeError(
        "Manifest references differ from frozen labels."
    )

print("\nDevelopment images:", len(trainval_df))

print(
    "Development patients:",
    trainval_df["patient_id"].nunique(),
)

print(
    "Frozen folds:",
    [
        int(value)
        for value in sorted(observed_folds)
    ],
)

print(
    "Primary patient reference:",
    PRIMARY_REFERENCE_RULE,
)


# ============================================================
# 14. Dataset and transforms
# ============================================================

class LIMUCDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform=None,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image_path = row["path"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,

            "label": torch.tensor(
                int(row["label"]),
                dtype=torch.long,
            ),

            "patient_id": torch.tensor(
                int(row["patient_id"]),
                dtype=torch.long,
            ),

            "patient_y_true": torch.tensor(
                int(row["patient_y_true"]),
                dtype=torch.long,
            ),

            "path": image_path,
        }


def build_train_transform():
    image_size = TRAINING_CONFIG[
        "image_size"
    ]

    return transforms.Compose([
        transforms.RandomResizedCrop(
            size=(
                image_size,
                image_size,
            ),
            scale=(
                0.75,
                1.00,
            ),
            ratio=(
                0.90,
                1.10,
            ),
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=8
        ),

        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.02,
        ),

        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(
                0.1,
                1.2,
            ),
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],
            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


def build_eval_transform():
    image_size = TRAINING_CONFIG[
        "image_size"
    ]

    return transforms.Compose([
        transforms.Resize(
            (
                image_size,
                image_size,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],
            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


# ============================================================
# 15. Metrics and aggregation
# ============================================================

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


def compute_metrics(
    y_true,
    y_pred,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(y_true) == 0:
        raise ValueError(
            "Cannot calculate metrics on an empty set."
        )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred have different lengths."
        )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
        weights="quadratic",
    )

    return {
        "qwk": (
            float(qwk)
            if np.isfinite(qwk)
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
                average="macro",
                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def make_confusion_df(
    y_true,
    y_pred,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
    )

    return pd.DataFrame(
        matrix,
        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],
        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


def aggregate_mean_probabilities(
    image_prediction_df: pd.DataFrame,
):
    required_columns = {
        "patient_id",
        "patient_y_true",
        "path",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        - set(image_prediction_df.columns)
    )

    if missing_columns:
        raise ValueError(
            "Prediction dataframe is missing columns: "
            f"{sorted(missing_columns)}"
        )

    reference_counts = (
        image_prediction_df
        .groupby("patient_id")[
            "patient_y_true"
        ]
        .nunique()
    )

    if reference_counts.max() != 1:
        raise RuntimeError(
            "A patient has inconsistent reference labels."
        )

    patient_probability_df = (
        image_prediction_df
        .groupby(
            "patient_id",
            as_index=False,
        )[PROBABILITY_COLUMNS]
        .mean()
    )

    patient_metadata_df = (
        image_prediction_df
        .groupby(
            "patient_id",
            as_index=False,
        )
        .agg(
            y_true=(
                "patient_y_true",
                "first",
            ),

            n_images=(
                "path",
                "size",
            ),
        )
    )

    patient_df = (
        patient_metadata_df
        .merge(
            patient_probability_df,
            on="patient_id",
            how="inner",
            validate="one_to_one",
        )
    )

    patient_df["patient_id"] = (
        patient_df["patient_id"]
        .astype(int)
    )

    patient_df["y_true"] = (
        patient_df["y_true"]
        .astype(int)
    )

    patient_df["n_images"] = (
        patient_df["n_images"]
        .astype(int)
    )

    patient_df["y_pred"] = (
        patient_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy()
        .argmax(axis=1)
        .astype(int)
    )

    return (
        patient_df
        .sort_values("patient_id")
        .reset_index(drop=True)
    )


# ============================================================
# 16. Prediction validation
# ============================================================

def validate_image_predictions(
    prediction_df: pd.DataFrame,
    expected_df: pd.DataFrame,
    expected_seed=None,
    expected_fold=None,
):
    prediction_df = prediction_df.copy()

    required_columns = {
        "path",
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        - set(prediction_df.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Prediction file is missing columns: "
            f"{sorted(missing_columns)}"
        )

    prediction_df["path"] = (
        prediction_df["path"]
        .astype(str)
    )

    for integer_column in [
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
    ]:
        prediction_df[integer_column] = (
            prediction_df[integer_column]
            .astype(int)
        )

    if prediction_df["path"].duplicated().any():
        raise RuntimeError(
            "Duplicate paths exist in a prediction file."
        )

    if len(prediction_df) != len(expected_df):
        raise RuntimeError(
            "Prediction row count differs from expected.\n"
            f"Expected: {len(expected_df)}\n"
            f"Observed: {len(prediction_df)}"
        )

    probability_array = (
        prediction_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    if not np.isfinite(
        probability_array
    ).all():
        raise RuntimeError(
            "Probabilities contain NaN or infinity."
        )

    probability_sums = (
        probability_array.sum(axis=1)
    )

    if not np.allclose(
        probability_sums,
        1.0,
        atol=2e-3,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Probability rows do not sum to one."
        )

    prediction_df["y_pred"] = (
        probability_array
        .argmax(axis=1)
        .astype(int)
    )

    expected_metadata = (
        expected_df[
            [
                "path",
                "patient_id",
                "label",
                "patient_y_true",
            ]
        ]
        .rename(
            columns={
                "patient_id": (
                    "patient_id_expected"
                ),
                "label": (
                    "y_true_expected"
                ),
                "patient_y_true": (
                    "patient_y_true_expected"
                ),
            }
        )
    )

    validation_df = (
        prediction_df[
            [
                "path",
                "patient_id",
                "y_true",
                "patient_y_true",
            ]
        ]
        .merge(
            expected_metadata,
            on="path",
            how="inner",
            validate="one_to_one",
        )
    )

    if len(validation_df) != len(expected_df):
        raise RuntimeError(
            "Prediction paths do not match expected images."
        )

    if not (
        validation_df["patient_id"]
        ==
        validation_df[
            "patient_id_expected"
        ]
    ).all():
        raise RuntimeError(
            "Prediction patient IDs differ from the manifest."
        )

    if not (
        validation_df["y_true"]
        ==
        validation_df[
            "y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            "Prediction labels differ from the manifest."
        )

    if not (
        validation_df["patient_y_true"]
        ==
        validation_df[
            "patient_y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            "Prediction patient references differ from "
            "the frozen reference."
        )

    if expected_seed is not None:
        prediction_df["training_seed"] = int(
            expected_seed
        )

    if expected_fold is not None:
        prediction_df["fold"] = int(
            expected_fold
        )

    return (
        prediction_df
        .sort_values("path")
        .reset_index(drop=True)
    )


# ============================================================
# 17. Ordinal losses
# ============================================================

class ExpectedValueOrdinalMAELoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
    ):
        super().__init__()

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        expected_class = (
            probabilities
            *
            self.class_values.unsqueeze(0)
        ).sum(dim=1)

        return torch.abs(
            expected_class
            - targets.float()
        ).mean()


class ExpectedOrdinalDistanceLoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
        alpha: float,
    ):
        super().__init__()

        self.alpha = float(alpha)

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        targets = (
            targets
            .float()
            .unsqueeze(1)
        )

        distances = torch.abs(
            targets
            - self.class_values.unsqueeze(0)
        ) ** self.alpha

        return (
            probabilities
            * distances
        ).sum(dim=1).mean()


class CombinedLoss(nn.Module):
    def __init__(
        self,
        loss_type: str,
        ce_weight,
        ordinal_loss_weight: float,
        ordinal_alpha: float,
    ):
        super().__init__()

        self.loss_type = loss_type

        self.ordinal_loss_weight = float(
            ordinal_loss_weight
        )

        self.cross_entropy = nn.CrossEntropyLoss(
            weight=ce_weight,
            label_smoothing=(
                TRAINING_CONFIG[
                    "label_smoothing"
                ]
            ),
        )

        if (
            loss_type
            ==
            "ce_expected_value_mae"
        ):
            self.ordinal_loss = (
                ExpectedValueOrdinalMAELoss(
                    num_classes=(
                        TRAINING_CONFIG[
                            "num_classes"
                        ]
                    )
                )
            )

        elif (
            loss_type
            ==
            "ce_expected_distance"
        ):
            self.ordinal_loss = (
                ExpectedOrdinalDistanceLoss(
                    num_classes=(
                        TRAINING_CONFIG[
                            "num_classes"
                        ]
                    ),
                    alpha=ordinal_alpha,
                )
            )

        else:
            raise ValueError(
                f"Unsupported loss type: {loss_type}"
            )

    def forward(
        self,
        logits,
        targets,
    ):
        ce_loss = self.cross_entropy(
            logits,
            targets,
        )

        ordinal_loss = self.ordinal_loss(
            logits,
            targets,
        )

        return (
            ce_loss
            +
            self.ordinal_loss_weight
            * ordinal_loss
        )


# ============================================================
# 18. Scheduler and AMP
# ============================================================

def build_scheduler(
    optimizer,
):
    warmup_epochs = TRAINING_CONFIG[
        "warmup_epochs"
    ]

    total_epochs = TRAINING_CONFIG[
        "epochs"
    ]

    min_lr_ratio = TRAINING_CONFIG[
        "min_lr_ratio"
    ]

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (
                float(epoch + 1)
                /
                float(
                    max(
                        1,
                        warmup_epochs,
                    )
                )
            )

        progress = (
            float(
                epoch
                - warmup_epochs
            )
            /
            float(
                max(
                    1,
                    total_epochs
                    - warmup_epochs,
                )
            )
        )

        cosine_value = 0.5 * (
            1.0
            +
            math.cos(
                math.pi
                * progress
            )
        )

        return max(
            min_lr_ratio,
            cosine_value,
        )

    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda,
    )


def create_grad_scaler(
    enabled: bool,
):
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=enabled
        )


def autocast_context(
    enabled: bool,
):
    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.autocast(
            enabled=enabled
        )


# ============================================================
# 19. Training and evaluation
# ============================================================

def calculate_class_weights(
    training_df: pd.DataFrame,
    device,
):
    class_counts = (
        training_df["label"]
        .value_counts()
        .reindex(
            range(
                TRAINING_CONFIG[
                    "num_classes"
                ]
            ),
            fill_value=0,
        )
        .sort_index()
    )

    if (class_counts == 0).any():
        missing_classes = (
            class_counts[
                class_counts == 0
            ]
            .index
            .tolist()
        )

        raise RuntimeError(
            "Training fold is missing classes: "
            f"{missing_classes}"
        )

    total_images = int(
        class_counts.sum()
    )

    class_weights = (
        total_images
        /
        (
            TRAINING_CONFIG[
                "num_classes"
            ]
            * class_counts
        )
    )

    class_weight_tensor = torch.tensor(
        class_weights.to_numpy(),
        dtype=torch.float32,
        device=device,
    )

    return (
        class_weight_tensor,

        {
            str(key): int(value)
            for key, value
            in class_counts.items()
        },

        {
            str(key): float(value)
            for key, value
            in class_weights.items()
        },
    )


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
):
    model.train()

    total_loss = 0.0

    y_true = []
    y_pred = []

    use_amp = (
        TRAINING_CONFIG[
            "use_mixed_precision"
        ]
        and
        device.type == "cuda"
        and
        scaler.is_enabled()
    )

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels = batch["label"].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(images)

            loss = criterion(
                logits,
                labels,
            )

        if use_amp:
            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                TRAINING_CONFIG[
                    "grad_clip_norm"
                ],
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                TRAINING_CONFIG[
                    "grad_clip_norm"
                ],
            )

            optimizer.step()

        total_loss += (
            float(loss.item())
            * images.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics["loss"] = (
        total_loss
        / len(loader.dataset)
    )

    return metrics


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0

    image_y_true = []
    image_y_pred = []
    image_probabilities = []

    patient_ids = []
    patient_y_true = []
    image_paths = []

    use_amp = (
        TRAINING_CONFIG[
            "use_mixed_precision"
        ]
        and
        device.type == "cuda"
    )

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels = batch["label"].to(
            device,
            non_blocking=True,
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(images)

            loss = criterion(
                logits,
                labels,
            )

            probability_list = [
                torch.softmax(
                    logits,
                    dim=1,
                )
            ]

            if (
                TRAINING_CONFIG["use_tta"]
                and
                TRAINING_CONFIG[
                    "tta_horizontal_flip"
                ]
            ):
                flipped_logits = model(
                    torch.flip(
                        images,
                        dims=[3],
                    )
                )

                probability_list.append(
                    torch.softmax(
                        flipped_logits,
                        dim=1,
                    )
                )

            probabilities = torch.stack(
                probability_list,
                dim=0,
            ).mean(dim=0)

        predictions = probabilities.argmax(
            dim=1
        )

        total_loss += (
            float(loss.item())
            * images.size(0)
        )

        image_y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        image_y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        image_probabilities.extend(
            probabilities.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        patient_ids.extend(
            batch["patient_id"]
            .cpu()
            .numpy()
            .tolist()
        )

        patient_y_true.extend(
            batch["patient_y_true"]
            .cpu()
            .numpy()
            .tolist()
        )

        image_paths.extend(
            batch["path"]
        )

    image_metrics = compute_metrics(
        image_y_true,
        image_y_pred,
    )

    image_metrics["loss"] = (
        total_loss
        / len(loader.dataset)
    )

    prediction_df = pd.DataFrame({
        "path": image_paths,
        "patient_id": patient_ids,
        "y_true": image_y_true,
        "patient_y_true": patient_y_true,
        "y_pred": image_y_pred,
    })

    prediction_df = pd.concat(
        [
            prediction_df,

            pd.DataFrame(
                image_probabilities,
                columns=PROBABILITY_COLUMNS,
            ),
        ],
        axis=1,
    )

    patient_df = (
        aggregate_mean_probabilities(
            prediction_df
        )
    )

    patient_metrics = compute_metrics(
        patient_df["y_true"],
        patient_df["y_pred"],
    )

    return (
        image_metrics,
        patient_metrics,
        prediction_df,
        patient_df,
    )


# ============================================================
# 20. Patient-primary checkpoint comparison
# ============================================================

def safe_metric_for_max(value):
    if (
        value is None
        or not np.isfinite(value)
    ):
        return -np.inf

    return float(value)


def safe_metric_for_min(value):
    if (
        value is None
        or not np.isfinite(value)
    ):
        return np.inf

    return float(value)


def is_better_checkpoint(
    candidate_patient_metrics,
    best_patient_metrics,
    candidate_image_metrics,
    best_image_metrics,
):
    if best_patient_metrics is None:
        return True

    candidate_tuple = (
        safe_metric_for_max(
            candidate_patient_metrics[
                "qwk"
            ]
        ),

        -safe_metric_for_min(
            candidate_patient_metrics[
                "mae"
            ]
        ),

        safe_metric_for_max(
            candidate_patient_metrics[
                "macro_f1"
            ]
        ),

        safe_metric_for_max(
            candidate_image_metrics[
                "qwk"
            ]
        ),
    )

    best_tuple = (
        safe_metric_for_max(
            best_patient_metrics[
                "qwk"
            ]
        ),

        -safe_metric_for_min(
            best_patient_metrics[
                "mae"
            ]
        ),

        safe_metric_for_max(
            best_patient_metrics[
                "macro_f1"
            ]
        ),

        safe_metric_for_max(
            best_image_metrics[
                "qwk"
            ]
        ),
    )

    return candidate_tuple > best_tuple


# ============================================================
# 21. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )


# ============================================================
# 22. Completed-fold loader
# ============================================================

def load_completed_fold(
    fold_directory: Path,
    expected_fold_fingerprint: str,
    expected_validation_df: pd.DataFrame,
):
    fold_result_path = (
        fold_directory
        / "fold_result.json"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    required_paths = [
        fold_result_path,
        image_prediction_path,
        patient_prediction_path,
        best_checkpoint_path,
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return None

    with open(
        fold_result_path,
        "r",
        encoding="utf-8",
    ) as file:
        fold_result = json.load(file)

    if (
        fold_result.get(
            "fold_fingerprint"
        )
        !=
        expected_fold_fingerprint
    ):
        raise RuntimeError(
            "A completed fold exists, but its configuration "
            "does not match the current plan:\n"
            f"{fold_directory}"
        )

    prediction_df = pd.read_csv(
        image_prediction_path
    )

    prediction_df = validate_image_predictions(
        prediction_df=prediction_df,
        expected_df=expected_validation_df,
        expected_seed=int(
            fold_result["training_seed"]
        ),
        expected_fold=int(
            fold_result["fold"]
        ),
    )

    patient_prediction_df = (
        aggregate_mean_probabilities(
            prediction_df
        )
    )

    patient_prediction_df["fold"] = int(
        fold_result["fold"]
    )

    patient_prediction_df[
        "training_seed"
    ] = int(
        fold_result["training_seed"]
    )

    print(
        "\nLoaded completed fold:",
        fold_result[
            "candidate_name"
        ],
        "| seed",
        fold_result[
            "training_seed"
        ],
        "| fold",
        fold_result["fold"],
    )

    return {
        "fold_result": fold_result,
        "image_predictions": prediction_df,
        "patient_predictions": (
            patient_prediction_df
        ),
    }


# ============================================================
# 23. Run or resume one fold
# ============================================================

def run_or_resume_fold(
    candidate_name: str,
    candidate_config: dict,
    training_seed: int,
    fold: int,
):
    fold_seed = (
        int(training_seed)
        + int(fold)
    )

    seed_directory = (
        OUTPUT_ROOT
        / candidate_name
        / f"seed_{training_seed}"
    )

    fold_directory = (
        seed_directory
        / f"fold_{fold}"
    )

    fold_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    fold_configuration = {
        "protocol_version": PROTOCOL_VERSION,

        "confirmation_plan_fingerprint": (
            confirmation_plan_fingerprint
        ),

        "candidate_name": candidate_name,

        "candidate_config": (
            candidate_config
        ),

        "training_seed": int(
            training_seed
        ),

        "fold": int(fold),

        "fold_seed": int(
            fold_seed
        ),

        "training_config": (
            TRAINING_CONFIG
        ),
    }

    fold_fingerprint = (
        dictionary_sha256(
            fold_configuration
        )
    )

    training_df = (
        trainval_df[
            trainval_df["cv_fold"]
            !=
            fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    validation_df = (
        trainval_df[
            trainval_df["cv_fold"]
            ==
            fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    patient_overlap = (
        set(
            training_df[
                "patient_id"
            ].unique()
        )
        &
        set(
            validation_df[
                "patient_id"
            ].unique()
        )
    )

    if patient_overlap:
        raise RuntimeError(
            f"Patient leakage detected in fold {fold}."
        )

    if RESUME_COMPLETED_FOLDS:
        completed_fold = load_completed_fold(
            fold_directory=(
                fold_directory
            ),

            expected_fold_fingerprint=(
                fold_fingerprint
            ),

            expected_validation_df=(
                validation_df
            ),
        )

        if completed_fold is not None:
            return completed_fold

    print("\n" + "=" * 100)

    print(
        f"{candidate_name} | "
        f"seed={training_seed} | "
        f"fold={fold + 1}/"
        f"{TRAINING_CONFIG['num_folds']}"
    )

    print("=" * 100)

    print(
        "Training images:",
        len(training_df),
    )

    print(
        "Validation images:",
        len(validation_df),
    )

    print(
        "Training patients:",
        training_df[
            "patient_id"
        ].nunique(),
    )

    print(
        "Validation patients:",
        validation_df[
            "patient_id"
        ].nunique(),
    )

    validation_patient_distribution = (
        validation_df[
            [
                "patient_id",
                "patient_y_true",
            ]
        ]
        .drop_duplicates()
        ["patient_y_true"]
        .value_counts()
        .reindex(
            range(
                TRAINING_CONFIG[
                    "num_classes"
                ]
            ),
            fill_value=0,
        )
        .sort_index()
    )

    for mayo_class in range(
        TRAINING_CONFIG[
            "num_classes"
        ]
    ):
        print(
            f"Validation Mayo {mayo_class} patients:",
            int(
                validation_patient_distribution.loc[
                    mayo_class
                ]
            ),
        )

    training_dataset = LIMUCDataset(
        training_df,
        transform=build_train_transform(),
    )

    validation_dataset = LIMUCDataset(
        validation_df,
        transform=build_eval_transform(),
    )

    training_generator = (
        torch.Generator()
    )

    validation_generator = (
        torch.Generator()
    )

    training_generator.manual_seed(
        fold_seed
    )

    validation_generator.manual_seed(
        fold_seed + 100000
    )

    training_loader = DataLoader(
        training_dataset,

        batch_size=TRAINING_CONFIG[
            "batch_size"
        ],

        shuffle=True,

        num_workers=TRAINING_CONFIG[
            "num_workers"
        ],

        pin_memory=(
            device.type == "cuda"
        ),

        drop_last=False,

        worker_init_fn=seed_worker,

        generator=training_generator,

        persistent_workers=False,
    )

    validation_loader = DataLoader(
        validation_dataset,

        batch_size=TRAINING_CONFIG[
            "batch_size"
        ],

        shuffle=False,

        num_workers=TRAINING_CONFIG[
            "num_workers"
        ],

        pin_memory=(
            device.type == "cuda"
        ),

        drop_last=False,

        worker_init_fn=seed_worker,

        generator=validation_generator,

        persistent_workers=False,
    )

    last_checkpoint_path = (
        fold_directory
        / "last_epoch_checkpoint.pt"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    history_path = (
        fold_directory
        / "history.csv"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    fold_result_path = (
        fold_directory
        / "fold_result.json"
    )

    resume_payload = None

    if (
        RESUME_INTERRUPTED_FOLDS
        and
        last_checkpoint_path.exists()
    ):
        try:
            # Critical fix: load the resume checkpoint on CPU.
            resume_payload = safe_torch_load(
                last_checkpoint_path,
                map_location="cpu",
            )

            if (
                resume_payload.get(
                    "fold_fingerprint"
                )
                !=
                fold_fingerprint
            ):
                raise RuntimeError(
                    "Interrupted checkpoint belongs "
                    "to another configuration."
                )

            print(
                "\nFound resumable checkpoint:",
                last_checkpoint_path,
            )

        except Exception as error:
            corrupt_path = (
                last_checkpoint_path
                .with_name(
                    "last_epoch_checkpoint_corrupt_"
                    +
                    datetime.now(
                        timezone.utc
                    ).strftime(
                        "%Y%m%d_%H%M%S"
                    )
                    +
                    ".pt"
                )
            )

            os.replace(
                last_checkpoint_path,
                corrupt_path,
            )

            print(
                "\nThe resume checkpoint could not be loaded."
            )

            print(
                "It was moved to:",
                corrupt_path,
            )

            print(
                "This fold will restart from epoch 1."
            )

            print(
                "Loading error:",
                repr(error),
            )

            resume_payload = None

    seed_everything(
        fold_seed
    )

    model = timm.create_model(
        TRAINING_CONFIG[
            "model_name"
        ],

        pretrained=(
            TRAINING_CONFIG[
                "pretrained"
            ]
            if resume_payload is None
            else False
        ),

        num_classes=TRAINING_CONFIG[
            "num_classes"
        ],

        drop_rate=TRAINING_CONFIG[
            "drop_rate"
        ],

        drop_path_rate=TRAINING_CONFIG[
            "drop_path_rate"
        ],
    ).to(device)

    pretrained_cfg = copy.deepcopy(
        getattr(
            model,
            "pretrained_cfg",
            {},
        )
    )

    (
        class_weight_tensor,
        training_class_counts,
        training_class_weights,
    ) = calculate_class_weights(
        training_df=training_df,
        device=device,
    )

    criterion = CombinedLoss(
        loss_type=(
            candidate_config[
                "loss_type"
            ]
        ),

        ce_weight=(
            class_weight_tensor
            if TRAINING_CONFIG[
                "use_class_weights_in_ce"
            ]
            else None
        ),

        ordinal_loss_weight=(
            candidate_config[
                "ordinal_loss_weight"
            ]
        ),

        ordinal_alpha=(
            candidate_config[
                "ordinal_alpha"
            ]
        ),
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),

        lr=TRAINING_CONFIG[
            "learning_rate"
        ],

        weight_decay=TRAINING_CONFIG[
            "weight_decay"
        ],
    )

    scheduler = build_scheduler(
        optimizer
    )

    scaler = create_grad_scaler(
        enabled=(
            TRAINING_CONFIG[
                "use_mixed_precision"
            ]
            and
            device.type == "cuda"
        )
    )

    start_epoch = 1
    best_epoch = -1

    best_patient_metrics = None
    best_image_metrics = None

    patience_counter = 0
    history = []

    if resume_payload is not None:
        model.load_state_dict(
            resume_payload[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )

        optimizer_to_device(
            optimizer,
            device,
        )

        scheduler.load_state_dict(
            resume_payload[
                "scheduler_state_dict"
            ]
        )

        scaler_state = resume_payload.get(
            "scaler_state_dict"
        )

        if scaler_state is not None:
            scaler.load_state_dict(
                scaler_state
            )

        best_epoch = int(
            resume_payload[
                "best_epoch"
            ]
        )

        best_patient_metrics = (
            resume_payload[
                "best_patient_metrics"
            ]
        )

        best_image_metrics = (
            resume_payload[
                "best_image_metrics"
            ]
        )

        patience_counter = int(
            resume_payload[
                "patience_counter"
            ]
        )

        history = resume_payload.get(
            "history",
            [],
        )

        completed_epoch = int(
            resume_payload[
                "completed_epoch"
            ]
        )

        start_epoch = (
            completed_epoch + 1
        )

        # Critical fix: CPU generators require CPU ByteTensor.
        training_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "training_generator_state"
                ],
                state_name=(
                    "Training DataLoader generator state"
                ),
            )
        )

        validation_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "validation_generator_state"
                ],
                state_name=(
                    "Validation DataLoader generator state"
                ),
            )
        )

        restore_rng_state(
            resume_payload[
                "rng_state"
            ]
        )

        pretrained_cfg = resume_payload.get(
            "pretrained_cfg",
            pretrained_cfg,
        )

        # Validate or repair the separate best checkpoint.
        repair_best_checkpoint = False

        if not best_checkpoint_path.exists():
            repair_best_checkpoint = True

        else:
            try:
                existing_best = safe_torch_load(
                    best_checkpoint_path,
                    map_location="cpu",
                )

                if (
                    existing_best.get(
                        "fold_fingerprint"
                    )
                    !=
                    fold_fingerprint
                ):
                    repair_best_checkpoint = True

                elif (
                    int(
                        existing_best.get(
                            "best_epoch",
                            -1,
                        )
                    )
                    !=
                    best_epoch
                ):
                    repair_best_checkpoint = True

                del existing_best

            except Exception:
                repair_best_checkpoint = True

        if repair_best_checkpoint:
            if best_epoch == completed_epoch:
                repaired_best_payload = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "confirmation_plan_fingerprint": (
                        confirmation_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "candidate_name": (
                        candidate_name
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "best_epoch": int(
                        best_epoch
                    ),

                    "model_state_dict": {
                        key: value.detach().cpu()

                        for key, value in (
                            resume_payload[
                                "model_state_dict"
                            ].items()
                        )
                    },

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),
                }

                atomic_torch_save(
                    repaired_best_payload,
                    best_checkpoint_path,
                )

                print(
                    "Repaired best checkpoint from "
                    "the resume checkpoint."
                )

            else:
                raise RuntimeError(
                    "The best checkpoint is missing or "
                    "inconsistent and cannot be reconstructed.\n"
                    f"Candidate: {candidate_name}\n"
                    f"Seed: {training_seed}\n"
                    f"Fold: {fold}\n"
                    f"Best epoch: {best_epoch}\n"
                    f"Completed epoch: {completed_epoch}"
                )

        print(
            "Resuming from completed epoch:",
            completed_epoch,
        )

        print(
            "Next epoch:",
            start_epoch,
        )

        print(
            "Current patience:",
            patience_counter,
            "/",
            TRAINING_CONFIG[
                "early_stop_patience"
            ],
        )

    training_required = (
        start_epoch
        <=
        TRAINING_CONFIG[
            "epochs"
        ]
        and
        patience_counter
        <
        TRAINING_CONFIG[
            "early_stop_patience"
        ]
    )

    if training_required:
        for epoch in range(
            start_epoch,
            TRAINING_CONFIG[
                "epochs"
            ] + 1,
        ):
            training_metrics = train_one_epoch(
                model=model,
                loader=training_loader,
                criterion=criterion,
                optimizer=optimizer,
                scaler=scaler,
                device=device,
            )

            (
                validation_image_metrics,
                validation_patient_metrics,
                _,
                _,
            ) = evaluate(
                model=model,
                loader=validation_loader,
                criterion=criterion,
                device=device,
            )

            current_lr = float(
                optimizer.param_groups[0][
                    "lr"
                ]
            )

            history_row = {
                "epoch": int(epoch),
                "lr": current_lr,

                "training_loss": float(
                    training_metrics[
                        "loss"
                    ]
                ),

                "training_image_qwk": float(
                    training_metrics[
                        "qwk"
                    ]
                ),

                "training_image_mae": float(
                    training_metrics[
                        "mae"
                    ]
                ),

                "training_image_macro_f1": float(
                    training_metrics[
                        "macro_f1"
                    ]
                ),

                "training_image_accuracy": float(
                    training_metrics[
                        "accuracy"
                    ]
                ),

                "validation_loss": float(
                    validation_image_metrics[
                        "loss"
                    ]
                ),

                "validation_image_qwk": float(
                    validation_image_metrics[
                        "qwk"
                    ]
                ),

                "validation_image_mae": float(
                    validation_image_metrics[
                        "mae"
                    ]
                ),

                "validation_image_macro_f1": float(
                    validation_image_metrics[
                        "macro_f1"
                    ]
                ),

                "validation_image_accuracy": float(
                    validation_image_metrics[
                        "accuracy"
                    ]
                ),

                "validation_patient_qwk": float(
                    validation_patient_metrics[
                        "qwk"
                    ]
                ),

                "validation_patient_mae": float(
                    validation_patient_metrics[
                        "mae"
                    ]
                ),

                "validation_patient_macro_f1": float(
                    validation_patient_metrics[
                        "macro_f1"
                    ]
                ),

                "validation_patient_accuracy": float(
                    validation_patient_metrics[
                        "accuracy"
                    ]
                ),
            }

            history.append(
                history_row
            )

            print(
                f"Epoch {epoch:02d}/"
                f"{TRAINING_CONFIG['epochs']} | "
                f"lr={current_lr:.6f} | "
                f"patient QWK="
                f"{validation_patient_metrics['qwk']:.4f} | "
                f"patient MAE="
                f"{validation_patient_metrics['mae']:.4f} | "
                f"patient F1="
                f"{validation_patient_metrics['macro_f1']:.4f} | "
                f"image QWK="
                f"{validation_image_metrics['qwk']:.4f}"
            )

            improved = is_better_checkpoint(
                candidate_patient_metrics=(
                    validation_patient_metrics
                ),

                best_patient_metrics=(
                    best_patient_metrics
                ),

                candidate_image_metrics=(
                    validation_image_metrics
                ),

                best_image_metrics=(
                    best_image_metrics
                ),
            )

            if improved:
                best_epoch = int(epoch)

                best_patient_metrics = (
                    copy.deepcopy(
                        validation_patient_metrics
                    )
                )

                best_image_metrics = (
                    copy.deepcopy(
                        validation_image_metrics
                    )
                )

                patience_counter = 0

                print(
                    "  New best validation checkpoint."
                )

            else:
                patience_counter += 1

                print(
                    "  No improvement. Patience "
                    f"{patience_counter}/"
                    f"{TRAINING_CONFIG['early_stop_patience']}"
                )

            atomic_csv_save(
                pd.DataFrame(history),
                history_path,
                index=False,
            )

            scheduler.step()

            resume_checkpoint_payload = {
                "protocol_version": (
                    PROTOCOL_VERSION
                ),

                "confirmation_plan_fingerprint": (
                    confirmation_plan_fingerprint
                ),

                "fold_fingerprint": (
                    fold_fingerprint
                ),

                "candidate_name": (
                    candidate_name
                ),

                "candidate_config": (
                    candidate_config
                ),

                "training_seed": int(
                    training_seed
                ),

                "fold": int(fold),

                "fold_seed": int(
                    fold_seed
                ),

                "completed_epoch": int(
                    epoch
                ),

                "best_epoch": int(
                    best_epoch
                ),

                "best_patient_metrics": (
                    best_patient_metrics
                ),

                "best_image_metrics": (
                    best_image_metrics
                ),

                "patience_counter": int(
                    patience_counter
                ),

                "history": history,

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),

                "scaler_state_dict": (
                    scaler.state_dict()
                ),

                "rng_state": (
                    capture_rng_state()
                ),

                "training_generator_state": (
                    training_generator
                    .get_state()
                    .cpu()
                    .contiguous()
                ),

                "validation_generator_state": (
                    validation_generator
                    .get_state()
                    .cpu()
                    .contiguous()
                ),

                "pretrained_cfg": (
                    pretrained_cfg
                ),

                "current_epoch_is_best": bool(
                    improved
                ),
            }

            atomic_torch_save(
                resume_checkpoint_payload,
                last_checkpoint_path,
            )

            print(
                "  Saved epoch-resume checkpoint."
            )

            if improved:
                best_checkpoint_payload = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "confirmation_plan_fingerprint": (
                        confirmation_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "candidate_name": (
                        candidate_name
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "best_epoch": int(
                        epoch
                    ),

                    "model_state_dict": {
                        key: value.detach().cpu()

                        for key, value in (
                            model.state_dict().items()
                        )
                    },

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),
                }

                atomic_torch_save(
                    best_checkpoint_payload,
                    best_checkpoint_path,
                )

                print(
                    "  Saved best-model checkpoint."
                )

            if (
                patience_counter
                >=
                TRAINING_CONFIG[
                    "early_stop_patience"
                ]
            ):
                print(
                    "Early stopping triggered."
                )
                break

    else:
        print(
            "Training already reached its stopping condition. "
            "Proceeding to best-checkpoint evaluation."
        )

    if not best_checkpoint_path.exists():
        raise RuntimeError(
            "No best checkpoint exists for:\n"
            f"{candidate_name}, "
            f"seed={training_seed}, "
            f"fold={fold}"
        )

    best_checkpoint = safe_torch_load(
        best_checkpoint_path,
        map_location="cpu",
    )

    if (
        best_checkpoint.get(
            "fold_fingerprint"
        )
        !=
        fold_fingerprint
    ):
        raise RuntimeError(
            "Best checkpoint fingerprint mismatch."
        )

    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )

    (
        final_image_metrics,
        final_patient_metrics,
        final_image_prediction_df,
        final_patient_prediction_df,
    ) = evaluate(
        model=model,
        loader=validation_loader,
        criterion=criterion,
        device=device,
    )

    final_image_prediction_df["fold"] = int(
        fold
    )

    final_image_prediction_df[
        "training_seed"
    ] = int(
        training_seed
    )

    final_image_prediction_df = (
        validate_image_predictions(
            prediction_df=(
                final_image_prediction_df
            ),

            expected_df=(
                validation_df
            ),

            expected_seed=(
                training_seed
            ),

            expected_fold=fold,
        )
    )

    final_patient_prediction_df["fold"] = int(
        fold
    )

    final_patient_prediction_df[
        "training_seed"
    ] = int(
        training_seed
    )

    atomic_csv_save(
        final_image_prediction_df,
        image_prediction_path,
        index=False,
    )

    atomic_csv_save(
        final_patient_prediction_df,
        patient_prediction_path,
        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            final_image_prediction_df[
                "y_true"
            ],
            final_image_prediction_df[
                "y_pred"
            ],
        ),

        fold_directory
        / "validation_image_confusion_matrix.csv",

        index=True,
    )

    atomic_csv_save(
        make_confusion_df(
            final_patient_prediction_df[
                "y_true"
            ],
            final_patient_prediction_df[
                "y_pred"
            ],
        ),

        fold_directory
        / "validation_patient_confusion_matrix.csv",

        index=True,
    )

    fold_result = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "confirmation_plan_fingerprint": (
            confirmation_plan_fingerprint
        ),

        "fold_fingerprint": (
            fold_fingerprint
        ),

        "candidate_name": (
            candidate_name
        ),

        "candidate_display_name": (
            candidate_config[
                "display_name"
            ]
        ),

        "training_seed": int(
            training_seed
        ),

        "fold": int(fold),

        "fold_seed": int(
            fold_seed
        ),

        "best_epoch": int(
            best_checkpoint[
                "best_epoch"
            ]
        ),

        "image_metrics": {
            key: float(value)

            for key, value in (
                final_image_metrics.items()
            )
        },

        "patient_metrics": {
            key: float(value)

            for key, value in (
                final_patient_metrics.items()
            )
        },

        "validation_patient_class_counts": {
            str(class_index): int(
                validation_patient_distribution.loc[
                    class_index
                ]
            )

            for class_index in range(
                TRAINING_CONFIG[
                    "num_classes"
                ]
            )
        },

        "training_image_class_counts": (
            training_class_counts
        ),

        "training_image_class_weights": (
            training_class_weights
        ),

        "best_checkpoint_path": str(
            best_checkpoint_path
        ),

        "history_path": str(
            history_path
        ),

        "image_prediction_path": str(
            image_prediction_path
        ),

        "patient_prediction_path": str(
            patient_prediction_path
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,
    }

    atomic_json_save(
        fold_result,
        fold_result_path,
    )

    if last_checkpoint_path.exists():
        last_checkpoint_path.unlink()

    print(
        "Fold completed and finalized:",
        candidate_name,
        "| seed",
        training_seed,
        "| fold",
        fold,
    )

    del model
    del criterion
    del optimizer
    del scheduler
    del scaler
    del best_checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "fold_result": fold_result,

        "image_predictions": (
            final_image_prediction_df
        ),

        "patient_predictions": (
            final_patient_prediction_df
        ),
    }


# ============================================================
# 24. Bootstrap utilities
# ============================================================

def stratified_patient_bootstrap(
    patient_df: pd.DataFrame,
    repetitions: int,
    seed: int,
):
    rng = np.random.default_rng(
        seed
    )

    class_frames = {
        class_index: (
            patient_df[
                patient_df["y_true"]
                ==
                class_index
            ]
            .reset_index(drop=True)
        )

        for class_index in range(
            TRAINING_CONFIG[
                "num_classes"
            ]
        )
    }

    for class_index, class_df in (
        class_frames.items()
    ):
        if len(class_df) == 0:
            raise RuntimeError(
                "Bootstrap cannot proceed because "
                f"class {class_index} is empty."
            )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_parts = []

        for class_index in range(
            TRAINING_CONFIG[
                "num_classes"
            ]
        ):
            class_df = class_frames[
                class_index
            ]

            sampled_indices = rng.integers(
                low=0,
                high=len(class_df),
                size=len(class_df),
            )

            sampled_parts.append(
                class_df.iloc[
                    sampled_indices
                ]
            )

        sampled_df = pd.concat(
            sampled_parts,
            ignore_index=True,
        )

        metrics = compute_metrics(
            sampled_df["y_true"],
            sampled_df["y_pred"],
        )

        bootstrap_rows.append({
            "bootstrap_index": int(
                bootstrap_index
            ),
            **metrics,
        })

    bootstrap_df = pd.DataFrame(
        bootstrap_rows
    )

    interval_rows = []

    for metric_name in [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]:
        values = (
            bootstrap_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        interval_rows.append({
            "metric": metric_name,

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "n_valid_bootstrap": int(
                len(values)
            ),
        })

    return (
        pd.DataFrame(
            interval_rows
        ),
        bootstrap_df,
    )


def align_patient_predictions(
    patient_df_a: pd.DataFrame,
    patient_df_b: pd.DataFrame,
):
    aligned_df = (
        patient_df_a[
            [
                "patient_id",
                "y_true",
                "y_pred",
            ]
        ]
        .rename(
            columns={
                "y_pred": "y_pred_a"
            }
        )
        .merge(
            patient_df_b[
                [
                    "patient_id",
                    "y_true",
                    "y_pred",
                ]
            ]
            .rename(
                columns={
                    "y_true": "y_true_b",
                    "y_pred": "y_pred_b",
                }
            ),
            on="patient_id",
            how="inner",
            validate="one_to_one",
        )
    )

    if len(aligned_df) != 478:
        raise RuntimeError(
            "Paired comparison did not align "
            "all 478 patients."
        )

    if not (
        aligned_df["y_true"]
        ==
        aligned_df["y_true_b"]
    ).all():
        raise RuntimeError(
            "Patient references differ between "
            "paired prediction sets."
        )

    return aligned_df.drop(
        columns=[
            "y_true_b"
        ]
    )


def paired_patient_bootstrap(
    patient_df_a: pd.DataFrame,
    patient_df_b: pd.DataFrame,
    repetitions: int,
    seed: int,
    candidate_a: str,
    candidate_b: str,
):
    aligned_df = align_patient_predictions(
        patient_df_a,
        patient_df_b,
    )

    point_metrics_a = compute_metrics(
        aligned_df["y_true"],
        aligned_df["y_pred_a"],
    )

    point_metrics_b = compute_metrics(
        aligned_df["y_true"],
        aligned_df["y_pred_b"],
    )

    class_frames = {
        class_index: (
            aligned_df[
                aligned_df["y_true"]
                ==
                class_index
            ]
            .reset_index(drop=True)
        )

        for class_index in range(
            TRAINING_CONFIG[
                "num_classes"
            ]
        )
    }

    rng = np.random.default_rng(
        seed
    )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_parts = []

        for class_index in range(
            TRAINING_CONFIG[
                "num_classes"
            ]
        ):
            class_df = class_frames[
                class_index
            ]

            sampled_indices = rng.integers(
                low=0,
                high=len(class_df),
                size=len(class_df),
            )

            sampled_parts.append(
                class_df.iloc[
                    sampled_indices
                ]
            )

        sampled_df = pd.concat(
            sampled_parts,
            ignore_index=True,
        )

        metrics_a = compute_metrics(
            sampled_df["y_true"],
            sampled_df["y_pred_a"],
        )

        metrics_b = compute_metrics(
            sampled_df["y_true"],
            sampled_df["y_pred_b"],
        )

        bootstrap_rows.append({
            "bootstrap_index": int(
                bootstrap_index
            ),

            "delta_qwk": (
                metrics_a["qwk"]
                -
                metrics_b["qwk"]
            ),

            "delta_mae": (
                metrics_a["mae"]
                -
                metrics_b["mae"]
            ),

            "delta_macro_f1": (
                metrics_a["macro_f1"]
                -
                metrics_b["macro_f1"]
            ),

            "delta_accuracy": (
                metrics_a["accuracy"]
                -
                metrics_b["accuracy"]
            ),
        })

    bootstrap_df = pd.DataFrame(
        bootstrap_rows
    )

    metric_mapping = {
        "qwk": "delta_qwk",
        "mae": "delta_mae",
        "macro_f1": "delta_macro_f1",
        "accuracy": "delta_accuracy",
    }

    summary_rows = []

    for metric_name, difference_column in (
        metric_mapping.items()
    ):
        values = (
            bootstrap_df[
                difference_column
            ]
            .dropna()
            .to_numpy()
        )

        point_difference = (
            point_metrics_a[metric_name]
            -
            point_metrics_b[metric_name]
        )

        probability_nonpositive = float(
            np.mean(values <= 0)
        )

        probability_nonnegative = float(
            np.mean(values >= 0)
        )

        bootstrap_two_sided_p = min(
            1.0,
            2.0
            *
            min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )

        summary_rows.append({
            "candidate_a": candidate_a,
            "candidate_b": candidate_b,

            "difference_definition": (
                f"{candidate_a} - {candidate_b}"
            ),

            "metric": metric_name,

            "point_difference": float(
                point_difference
            ),

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "bootstrap_two_sided_p": float(
                bootstrap_two_sided_p
            ),

            "direction_note": (
                "Positive favors candidate A"
                if metric_name in [
                    "qwk",
                    "macro_f1",
                    "accuracy",
                ]
                else
                "Negative favors candidate A"
            ),
        })

    return (
        pd.DataFrame(
            summary_rows
        ),
        bootstrap_df,
    )


# ============================================================
# 25. Build one complete seed result
# ============================================================

def build_seed_result(
    candidate_name: str,
    training_seed: int,
    oof_image_df: pd.DataFrame,
    source_type: str,
):
    candidate_config = CANDIDATES[
        candidate_name
    ]

    seed_fingerprint = dictionary_sha256({
        "protocol_version": PROTOCOL_VERSION,

        "confirmation_plan_fingerprint": (
            confirmation_plan_fingerprint
        ),

        "candidate_name": (
            candidate_name
        ),

        "candidate_config": (
            candidate_config
        ),

        "training_seed": int(
            training_seed
        ),

        "source_type": (
            source_type
        ),
    })

    seed_directory = (
        OUTPUT_ROOT
        / candidate_name
        / f"seed_{training_seed}"
    )

    seed_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_result_path = (
        seed_directory
        / "seed_result.json"
    )

    oof_image_path = (
        seed_directory
        / "oof_image_predictions.csv"
    )

    oof_patient_path = (
        seed_directory
        / "oof_patient_predictions.csv"
    )

    bootstrap_ci_path = (
        seed_directory
        / "oof_patient_bootstrap_ci.csv"
    )

    bootstrap_samples_path = (
        seed_directory
        / "oof_patient_bootstrap_samples.csv"
    )

    completed_paths = [
        seed_result_path,
        oof_image_path,
        oof_patient_path,
        bootstrap_ci_path,
        bootstrap_samples_path,
    ]

    if (
        SKIP_COMPLETED_SEEDS
        and
        all(
            path.exists()
            for path in completed_paths
        )
    ):
        with open(
            seed_result_path,
            "r",
            encoding="utf-8",
        ) as file:
            existing_seed_result = (
                json.load(file)
            )

        if (
            existing_seed_result.get(
                "seed_fingerprint"
            )
            !=
            seed_fingerprint
        ):
            raise RuntimeError(
                "Completed seed fingerprint mismatch:\n"
                f"{seed_directory}"
            )

        existing_image_df = pd.read_csv(
            oof_image_path
        )

        existing_image_df = (
            validate_image_predictions(
                prediction_df=(
                    existing_image_df
                ),

                expected_df=(
                    trainval_df
                ),

                expected_seed=(
                    training_seed
                ),

                expected_fold=None,
            )
        )

        existing_patient_df = (
            aggregate_mean_probabilities(
                existing_image_df
            )
        )

        print(
            "Loaded completed seed result:",
            candidate_name,
            "| seed",
            training_seed,
        )

        return (
            existing_seed_result,
            existing_image_df,
            existing_patient_df,
        )

    oof_image_df = validate_image_predictions(
        prediction_df=oof_image_df,
        expected_df=trainval_df,
        expected_seed=training_seed,
        expected_fold=None,
    )

    if len(oof_image_df) != 9590:
        raise RuntimeError(
            "OOF image count is not 9,590."
        )

    if (
        "fold" in oof_image_df.columns
        and
        oof_image_df
        .groupby("patient_id")["fold"]
        .nunique()
        .max()
        !=
        1
    ):
        raise RuntimeError(
            "A patient appears in multiple OOF folds."
        )

    oof_patient_df = (
        aggregate_mean_probabilities(
            oof_image_df
        )
    )

    if len(oof_patient_df) != 478:
        raise RuntimeError(
            "OOF patient count is not 478."
        )

    image_metrics = compute_metrics(
        oof_image_df["y_true"],
        oof_image_df["y_pred"],
    )

    patient_metrics = compute_metrics(
        oof_patient_df["y_true"],
        oof_patient_df["y_pred"],
    )

    (
        bootstrap_ci_df,
        bootstrap_samples_df,
    ) = stratified_patient_bootstrap(
        patient_df=(
            oof_patient_df
        ),

        repetitions=(
            TRAINING_CONFIG[
                "bootstrap_repetitions"
            ]
        ),

        seed=(
            TRAINING_CONFIG[
                "bootstrap_seed"
            ]
            +
            int(training_seed)
            +
            list(
                CANDIDATES.keys()
            ).index(candidate_name)
            *
            10000
        ),
    )

    point_estimate_map = {
        "qwk": (
            patient_metrics["qwk"]
        ),
        "mae": (
            patient_metrics["mae"]
        ),
        "macro_f1": (
            patient_metrics[
                "macro_f1"
            ]
        ),
        "accuracy": (
            patient_metrics[
                "accuracy"
            ]
        ),
    }

    bootstrap_ci_df.insert(
        1,
        "point_estimate",
        bootstrap_ci_df[
            "metric"
        ].map(
            point_estimate_map
        ),
    )

    atomic_csv_save(
        oof_image_df,
        oof_image_path,
        index=False,
    )

    atomic_csv_save(
        oof_patient_df,
        oof_patient_path,
        index=False,
    )

    atomic_csv_save(
        bootstrap_ci_df,
        bootstrap_ci_path,
        index=False,
    )

    atomic_csv_save(
        bootstrap_samples_df,
        bootstrap_samples_path,
        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_image_df["y_true"],
            oof_image_df["y_pred"],
        ),

        seed_directory
        / "oof_image_confusion_matrix.csv",

        index=True,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_patient_df["y_true"],
            oof_patient_df["y_pred"],
        ),

        seed_directory
        / "oof_patient_confusion_matrix.csv",

        index=True,
    )

    ci_lookup = (
        bootstrap_ci_df
        .set_index("metric")
    )

    seed_result = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "confirmation_plan_fingerprint": (
            confirmation_plan_fingerprint
        ),

        "seed_fingerprint": (
            seed_fingerprint
        ),

        "candidate_name": (
            candidate_name
        ),

        "display_name": (
            candidate_config[
                "display_name"
            ]
        ),

        "training_seed": int(
            training_seed
        ),

        "source_type": (
            source_type
        ),

        "oof_image_qwk": float(
            image_metrics["qwk"]
        ),

        "oof_image_mae": float(
            image_metrics["mae"]
        ),

        "oof_image_macro_f1": float(
            image_metrics[
                "macro_f1"
            ]
        ),

        "oof_image_accuracy": float(
            image_metrics[
                "accuracy"
            ]
        ),

        "oof_patient_qwk": float(
            patient_metrics["qwk"]
        ),

        "oof_patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "oof_patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "oof_patient_mae": float(
            patient_metrics["mae"]
        ),

        "oof_patient_mae_ci_low": float(
            ci_lookup.loc[
                "mae",
                "lower_95",
            ]
        ),

        "oof_patient_mae_ci_high": float(
            ci_lookup.loc[
                "mae",
                "upper_95",
            ]
        ),

        "oof_patient_macro_f1": float(
            patient_metrics[
                "macro_f1"
            ]
        ),

        "oof_patient_macro_f1_ci_low": float(
            ci_lookup.loc[
                "macro_f1",
                "lower_95",
            ]
        ),

        "oof_patient_macro_f1_ci_high": float(
            ci_lookup.loc[
                "macro_f1",
                "upper_95",
            ]
        ),

        "oof_patient_accuracy": float(
            patient_metrics[
                "accuracy"
            ]
        ),

        "oof_patient_accuracy_ci_low": float(
            ci_lookup.loc[
                "accuracy",
                "lower_95",
            ]
        ),

        "oof_patient_accuracy_ci_high": float(
            ci_lookup.loc[
                "accuracy",
                "upper_95",
            ]
        ),

        "oof_image_predictions_path": str(
            oof_image_path
        ),

        "oof_patient_predictions_path": str(
            oof_patient_path
        ),

        "bootstrap_ci_path": str(
            bootstrap_ci_path
        ),

        "bootstrap_samples_path": str(
            bootstrap_samples_path
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,
    }

    atomic_json_save(
        seed_result,
        seed_result_path,
    )

    print(
        "Completed seed result:",
        candidate_name,
        "| seed",
        training_seed,
    )

    return (
        seed_result,
        oof_image_df,
        oof_patient_df,
    )


# ============================================================
# 26. Import seed 42
# ============================================================

seed_results = {}
seed_image_predictions = {}
seed_patient_predictions = {}

for candidate_name in CANDIDATES:
    source_oof_path = Path(
        screening_source_files[
            candidate_name
        ]["oof_path"]
    )

    imported_oof_df = pd.read_csv(
        source_oof_path
    )

    (
        imported_seed_result,
        imported_image_df,
        imported_patient_df,
    ) = build_seed_result(
        candidate_name=(
            candidate_name
        ),

        training_seed=(
            IMPORTED_SCREENING_SEED
        ),

        oof_image_df=(
            imported_oof_df
        ),

        source_type=(
            "imported_from_notebook_01_screening"
        ),
    )

    seed_results[
        (
            candidate_name,
            IMPORTED_SCREENING_SEED,
        )
    ] = imported_seed_result

    seed_image_predictions[
        (
            candidate_name,
            IMPORTED_SCREENING_SEED,
        )
    ] = imported_image_df

    seed_patient_predictions[
        (
            candidate_name,
            IMPORTED_SCREENING_SEED,
        )
    ] = imported_patient_df


# ============================================================
# 27. Train or resume new seeds
# ============================================================

for candidate_name, candidate_config in (
    CANDIDATES.items()
):
    for training_seed in (
        NEW_CONFIRMATION_SEEDS
    ):
        seed_directory = (
            OUTPUT_ROOT
            / candidate_name
            / f"seed_{training_seed}"
        )

        completed_seed_result_path = (
            seed_directory
            / "seed_result.json"
        )

        completed_oof_image_path = (
            seed_directory
            / "oof_image_predictions.csv"
        )

        if (
            SKIP_COMPLETED_SEEDS
            and
            completed_seed_result_path.exists()
            and
            completed_oof_image_path.exists()
        ):
            completed_oof_df = pd.read_csv(
                completed_oof_image_path
            )

            (
                seed_result,
                seed_image_df,
                seed_patient_df,
            ) = build_seed_result(
                candidate_name=(
                    candidate_name
                ),

                training_seed=(
                    training_seed
                ),

                oof_image_df=(
                    completed_oof_df
                ),

                source_type=(
                    "new_confirmation_training"
                ),
            )

        else:
            fold_prediction_parts = []
            fold_summary_rows = []

            for fold in range(
                TRAINING_CONFIG[
                    "num_folds"
                ]
            ):
                fold_output = (
                    run_or_resume_fold(
                        candidate_name=(
                            candidate_name
                        ),

                        candidate_config=(
                            candidate_config
                        ),

                        training_seed=(
                            training_seed
                        ),

                        fold=fold,
                    )
                )

                fold_prediction_parts.append(
                    fold_output[
                        "image_predictions"
                    ]
                )

                fold_result = (
                    fold_output[
                        "fold_result"
                    ]
                )

                fold_summary_rows.append({
                    "candidate_name": (
                        candidate_name
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "best_epoch": int(
                        fold_result[
                            "best_epoch"
                        ]
                    ),

                    "validation_image_qwk": float(
                        fold_result[
                            "image_metrics"
                        ]["qwk"]
                    ),

                    "validation_image_mae": float(
                        fold_result[
                            "image_metrics"
                        ]["mae"]
                    ),

                    "validation_image_macro_f1": float(
                        fold_result[
                            "image_metrics"
                        ]["macro_f1"]
                    ),

                    "validation_image_accuracy": float(
                        fold_result[
                            "image_metrics"
                        ]["accuracy"]
                    ),

                    "validation_patient_qwk": float(
                        fold_result[
                            "patient_metrics"
                        ]["qwk"]
                    ),

                    "validation_patient_mae": float(
                        fold_result[
                            "patient_metrics"
                        ]["mae"]
                    ),

                    "validation_patient_macro_f1": float(
                        fold_result[
                            "patient_metrics"
                        ]["macro_f1"]
                    ),

                    "validation_patient_accuracy": float(
                        fold_result[
                            "patient_metrics"
                        ]["accuracy"]
                    ),
                })

            oof_image_df = (
                pd.concat(
                    fold_prediction_parts,
                    ignore_index=True,
                )
                .sort_values("path")
                .reset_index(drop=True)
            )

            atomic_csv_save(
                pd.DataFrame(
                    fold_summary_rows
                ),

                seed_directory
                / "fold_summary.csv",

                index=False,
            )

            (
                seed_result,
                seed_image_df,
                seed_patient_df,
            ) = build_seed_result(
                candidate_name=(
                    candidate_name
                ),

                training_seed=(
                    training_seed
                ),

                oof_image_df=(
                    oof_image_df
                ),

                source_type=(
                    "new_confirmation_training"
                ),
            )

        seed_results[
            (
                candidate_name,
                training_seed,
            )
        ] = seed_result

        seed_image_predictions[
            (
                candidate_name,
                training_seed,
            )
        ] = seed_image_df

        seed_patient_predictions[
            (
                candidate_name,
                training_seed,
            )
        ] = seed_patient_df


# ============================================================
# 28. Save per-seed results
# ============================================================

per_seed_results_df = pd.DataFrame(
    list(
        seed_results.values()
    )
)

per_seed_results_df = (
    per_seed_results_df
    .sort_values(
        [
            "candidate_name",
            "training_seed",
        ]
    )
    .reset_index(drop=True)
)

per_seed_results_path = (
    OUTPUT_ROOT
    / "vit_multiseed_confirmation_per_seed_results.csv"
)

atomic_csv_save(
    per_seed_results_df,
    per_seed_results_path,
    index=False,
)


# ============================================================
# 29. Mean ± SD across seeds
# ============================================================

metric_columns = [
    "oof_image_qwk",
    "oof_image_mae",
    "oof_image_macro_f1",
    "oof_image_accuracy",

    "oof_patient_qwk",
    "oof_patient_mae",
    "oof_patient_macro_f1",
    "oof_patient_accuracy",
]

summary_rows = []

for candidate_name in CANDIDATES:
    candidate_rows = (
        per_seed_results_df[
            per_seed_results_df[
                "candidate_name"
            ]
            ==
            candidate_name
        ]
    )

    observed_seeds = sorted(
        candidate_rows[
            "training_seed"
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    expected_seeds = sorted(
        ALL_CONFIRMATION_SEEDS
    )

    if observed_seeds != expected_seeds:
        raise RuntimeError(
            f"{candidate_name} does not have all seeds.\n"
            f"Expected: {expected_seeds}\n"
            f"Observed: {observed_seeds}"
        )

    summary_row = {
        "candidate_name": (
            candidate_name
        ),

        "display_name": (
            CANDIDATES[
                candidate_name
            ]["display_name"]
        ),

        "n_training_seeds": int(
            len(expected_seeds)
        ),
    }

    for metric_name in metric_columns:
        summary_row[
            f"{metric_name}_mean"
        ] = float(
            candidate_rows[
                metric_name
            ].mean()
        )

        summary_row[
            f"{metric_name}_std"
        ] = float(
            candidate_rows[
                metric_name
            ].std(ddof=1)
        )

    summary_rows.append(
        summary_row
    )

multiseed_summary_df = pd.DataFrame(
    summary_rows
)

multiseed_summary_ranked_df = (
    multiseed_summary_df
    .sort_values(
        by=[
            "oof_patient_qwk_mean",
            "oof_patient_mae_mean",
            "oof_patient_macro_f1_mean",
            "oof_image_qwk_mean",
        ],
        ascending=[
            False,
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

multiseed_summary_ranked_df.insert(
    0,
    "development_rank",
    np.arange(
        1,
        len(
            multiseed_summary_ranked_df
        ) + 1,
    ),
)

multiseed_summary_path = (
    OUTPUT_ROOT
    / "vit_multiseed_confirmation_summary.csv"
)

atomic_csv_save(
    multiseed_summary_ranked_df,
    multiseed_summary_path,
    index=False,
)


# ============================================================
# 30. Paired bootstrap by seed
# ============================================================

paired_by_seed_summary_parts = []
paired_by_seed_sample_parts = []

for training_seed in (
    ALL_CONFIRMATION_SEEDS
):
    (
        paired_summary_df,
        paired_samples_df,
    ) = paired_patient_bootstrap(
        patient_df_a=(
            seed_patient_predictions[
                (
                    CANDIDATE_A,
                    training_seed,
                )
            ]
        ),

        patient_df_b=(
            seed_patient_predictions[
                (
                    CANDIDATE_B,
                    training_seed,
                )
            ]
        ),

        repetitions=(
            TRAINING_CONFIG[
                "bootstrap_repetitions"
            ]
        ),

        seed=(
            TRAINING_CONFIG[
                "bootstrap_seed"
            ]
            +
            100000
            +
            int(training_seed)
        ),

        candidate_a=(
            CANDIDATE_A
        ),

        candidate_b=(
            CANDIDATE_B
        ),
    )

    paired_summary_df.insert(
        0,
        "training_seed",
        int(training_seed),
    )

    paired_samples_df.insert(
        0,
        "training_seed",
        int(training_seed),
    )

    paired_by_seed_summary_parts.append(
        paired_summary_df
    )

    paired_by_seed_sample_parts.append(
        paired_samples_df
    )

paired_by_seed_summary_df = pd.concat(
    paired_by_seed_summary_parts,
    ignore_index=True,
)

paired_by_seed_samples_df = pd.concat(
    paired_by_seed_sample_parts,
    ignore_index=True,
)

paired_by_seed_summary_path = (
    OUTPUT_ROOT
    / "vit_paired_bootstrap_by_seed_summary.csv"
)

paired_by_seed_samples_path = (
    OUTPUT_ROOT
    / "vit_paired_bootstrap_by_seed_samples.csv"
)

atomic_csv_save(
    paired_by_seed_summary_df,
    paired_by_seed_summary_path,
    index=False,
)

atomic_csv_save(
    paired_by_seed_samples_df,
    paired_by_seed_samples_path,
    index=False,
)


# ============================================================
# 31. Paired bootstrap of mean metrics across seeds
# ============================================================

def paired_multiseed_patient_bootstrap(
    candidate_a: str,
    candidate_b: str,
    seeds,
    repetitions: int,
    seed: int,
):
    base_df = (
        seed_patient_predictions[
            (
                candidate_a,
                seeds[0],
            )
        ][
            [
                "patient_id",
                "y_true",
            ]
        ]
        .copy()
    )

    for training_seed in seeds:
        candidate_a_df = (
            seed_patient_predictions[
                (
                    candidate_a,
                    training_seed,
                )
            ][
                [
                    "patient_id",
                    "y_true",
                    "y_pred",
                ]
            ]
            .rename(
                columns={
                    "y_true": (
                        f"y_true_a_{training_seed}"
                    ),

                    "y_pred": (
                        f"y_pred_a_{training_seed}"
                    ),
                }
            )
        )

        candidate_b_df = (
            seed_patient_predictions[
                (
                    candidate_b,
                    training_seed,
                )
            ][
                [
                    "patient_id",
                    "y_true",
                    "y_pred",
                ]
            ]
            .rename(
                columns={
                    "y_true": (
                        f"y_true_b_{training_seed}"
                    ),

                    "y_pred": (
                        f"y_pred_b_{training_seed}"
                    ),
                }
            )
        )

        base_df = (
            base_df
            .merge(
                candidate_a_df,
                on="patient_id",
                how="inner",
                validate="one_to_one",
            )
            .merge(
                candidate_b_df,
                on="patient_id",
                how="inner",
                validate="one_to_one",
            )
        )

        if not (
            base_df["y_true"]
            ==
            base_df[
                f"y_true_a_{training_seed}"
            ]
        ).all():
            raise RuntimeError(
                "Candidate A references differ "
                f"for seed {training_seed}."
            )

        if not (
            base_df["y_true"]
            ==
            base_df[
                f"y_true_b_{training_seed}"
            ]
        ).all():
            raise RuntimeError(
                "Candidate B references differ "
                f"for seed {training_seed}."
            )

        base_df = base_df.drop(
            columns=[
                f"y_true_a_{training_seed}",
                f"y_true_b_{training_seed}",
            ]
        )

    if len(base_df) != 478:
        raise RuntimeError(
            "Multi-seed comparison did not align "
            "all 478 patients."
        )

    metric_names = [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]

    point_means_a = {}
    point_means_b = {}

    for metric_name in metric_names:
        values_a = []
        values_b = []

        for training_seed in seeds:
            metrics_a = compute_metrics(
                base_df["y_true"],

                base_df[
                    f"y_pred_a_{training_seed}"
                ],
            )

            metrics_b = compute_metrics(
                base_df["y_true"],

                base_df[
                    f"y_pred_b_{training_seed}"
                ],
            )

            values_a.append(
                metrics_a[metric_name]
            )

            values_b.append(
                metrics_b[metric_name]
            )

        point_means_a[
            metric_name
        ] = float(
            np.mean(values_a)
        )

        point_means_b[
            metric_name
        ] = float(
            np.mean(values_b)
        )

    class_frames = {
        class_index: (
            base_df[
                base_df["y_true"]
                ==
                class_index
            ]
            .reset_index(drop=True)
        )

        for class_index in range(
            TRAINING_CONFIG[
                "num_classes"
            ]
        )
    }

    rng = np.random.default_rng(
        seed
    )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_parts = []

        for class_index in range(
            TRAINING_CONFIG[
                "num_classes"
            ]
        ):
            class_df = class_frames[
                class_index
            ]

            sampled_indices = rng.integers(
                low=0,
                high=len(class_df),
                size=len(class_df),
            )

            sampled_parts.append(
                class_df.iloc[
                    sampled_indices
                ]
            )

        sampled_df = pd.concat(
            sampled_parts,
            ignore_index=True,
        )

        seed_metrics_a = {
            name: []
            for name in metric_names
        }

        seed_metrics_b = {
            name: []
            for name in metric_names
        }

        for training_seed in seeds:
            metrics_a = compute_metrics(
                sampled_df["y_true"],

                sampled_df[
                    f"y_pred_a_{training_seed}"
                ],
            )

            metrics_b = compute_metrics(
                sampled_df["y_true"],

                sampled_df[
                    f"y_pred_b_{training_seed}"
                ],
            )

            for metric_name in metric_names:
                seed_metrics_a[
                    metric_name
                ].append(
                    metrics_a[
                        metric_name
                    ]
                )

                seed_metrics_b[
                    metric_name
                ].append(
                    metrics_b[
                        metric_name
                    ]
                )

        bootstrap_rows.append({
            "bootstrap_index": int(
                bootstrap_index
            ),

            "delta_mean_qwk": (
                np.mean(
                    seed_metrics_a["qwk"]
                )
                -
                np.mean(
                    seed_metrics_b["qwk"]
                )
            ),

            "delta_mean_mae": (
                np.mean(
                    seed_metrics_a["mae"]
                )
                -
                np.mean(
                    seed_metrics_b["mae"]
                )
            ),

            "delta_mean_macro_f1": (
                np.mean(
                    seed_metrics_a[
                        "macro_f1"
                    ]
                )
                -
                np.mean(
                    seed_metrics_b[
                        "macro_f1"
                    ]
                )
            ),

            "delta_mean_accuracy": (
                np.mean(
                    seed_metrics_a[
                        "accuracy"
                    ]
                )
                -
                np.mean(
                    seed_metrics_b[
                        "accuracy"
                    ]
                )
            ),
        })

    bootstrap_df = pd.DataFrame(
        bootstrap_rows
    )

    difference_columns = {
        "qwk": "delta_mean_qwk",
        "mae": "delta_mean_mae",
        "macro_f1": (
            "delta_mean_macro_f1"
        ),
        "accuracy": (
            "delta_mean_accuracy"
        ),
    }

    summary_rows = []

    for metric_name, difference_column in (
        difference_columns.items()
    ):
        values = (
            bootstrap_df[
                difference_column
            ]
            .dropna()
            .to_numpy()
        )

        point_difference = (
            point_means_a[metric_name]
            -
            point_means_b[metric_name]
        )

        probability_nonpositive = float(
            np.mean(values <= 0)
        )

        probability_nonnegative = float(
            np.mean(values >= 0)
        )

        bootstrap_two_sided_p = min(
            1.0,
            2.0
            *
            min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )

        summary_rows.append({
            "candidate_a": candidate_a,
            "candidate_b": candidate_b,

            "difference_definition": (
                "Mean metric across seeds for candidate A "
                "minus mean metric across seeds for candidate B"
            ),

            "metric": metric_name,

            "point_difference": float(
                point_difference
            ),

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "bootstrap_two_sided_p": float(
                bootstrap_two_sided_p
            ),

            "direction_note": (
                "Positive favors candidate A"
                if metric_name in [
                    "qwk",
                    "macro_f1",
                    "accuracy",
                ]
                else
                "Negative favors candidate A"
            ),
        })

    return (
        pd.DataFrame(
            summary_rows
        ),
        bootstrap_df,
    )


(
    paired_multiseed_summary_df,
    paired_multiseed_samples_df,
) = paired_multiseed_patient_bootstrap(
    candidate_a=CANDIDATE_A,
    candidate_b=CANDIDATE_B,

    seeds=ALL_CONFIRMATION_SEEDS,

    repetitions=(
        TRAINING_CONFIG[
            "bootstrap_repetitions"
        ]
    ),

    seed=(
        TRAINING_CONFIG[
            "bootstrap_seed"
        ]
        +
        200000
    ),
)

paired_multiseed_summary_path = (
    OUTPUT_ROOT
    / "vit_paired_multiseed_bootstrap_summary.csv"
)

paired_multiseed_samples_path = (
    OUTPUT_ROOT
    / "vit_paired_multiseed_bootstrap_samples.csv"
)

atomic_csv_save(
    paired_multiseed_summary_df,
    paired_multiseed_summary_path,
    index=False,
)

atomic_csv_save(
    paired_multiseed_samples_df,
    paired_multiseed_samples_path,
    index=False,
)


# ============================================================
# 32. Seed-ensemble predictions
# ============================================================

def create_seed_ensemble_image_predictions(
    candidate_name: str,
):
    first_seed_df = (
        seed_image_predictions[
            (
                candidate_name,
                ALL_CONFIRMATION_SEEDS[0],
            )
        ]
        .sort_values("path")
        .reset_index(drop=True)
    )

    ensemble_metadata = first_seed_df[
        [
            "path",
            "patient_id",
            "y_true",
            "patient_y_true",
        ]
    ].copy()

    probability_arrays = []

    for training_seed in (
        ALL_CONFIRMATION_SEEDS
    ):
        seed_df = (
            seed_image_predictions[
                (
                    candidate_name,
                    training_seed,
                )
            ]
            .sort_values("path")
            .reset_index(drop=True)
        )

        seed_metadata = seed_df[
            [
                "path",
                "patient_id",
                "y_true",
                "patient_y_true",
            ]
        ]

        try:
            pd.testing.assert_frame_equal(
                seed_metadata,
                ensemble_metadata,
                check_dtype=False,
            )

        except AssertionError as error:
            raise RuntimeError(
                "Seed prediction metadata do not align for "
                f"{candidate_name}, seed {training_seed}.\n"
                f"{error}"
            )

        probability_arrays.append(
            seed_df[
                PROBABILITY_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )

    ensemble_probability_array = (
        np.stack(
            probability_arrays,
            axis=0,
        )
        .mean(axis=0)
    )

    ensemble_df = (
        ensemble_metadata.copy()
    )

    for column_index, column_name in enumerate(
        PROBABILITY_COLUMNS
    ):
        ensemble_df[column_name] = (
            ensemble_probability_array[
                :,
                column_index,
            ]
        )

    ensemble_df["y_pred"] = (
        ensemble_probability_array
        .argmax(axis=1)
        .astype(int)
    )

    ensemble_df[
        "ensemble_description"
    ] = (
        "mean_probability_across_3_seeds"
    )

    return validate_image_predictions(
        prediction_df=ensemble_df,
        expected_df=trainval_df,
        expected_seed=None,
        expected_fold=None,
    )


seed_ensemble_results = {}
seed_ensemble_patient_predictions = {}

for candidate_index, candidate_name in enumerate(
    CANDIDATES
):
    ensemble_directory = (
        OUTPUT_ROOT
        / candidate_name
        / "seed_ensemble"
    )

    ensemble_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    ensemble_image_df = (
        create_seed_ensemble_image_predictions(
            candidate_name
        )
    )

    ensemble_patient_df = (
        aggregate_mean_probabilities(
            ensemble_image_df
        )
    )

    ensemble_image_metrics = compute_metrics(
        ensemble_image_df["y_true"],
        ensemble_image_df["y_pred"],
    )

    ensemble_patient_metrics = compute_metrics(
        ensemble_patient_df["y_true"],
        ensemble_patient_df["y_pred"],
    )

    (
        ensemble_ci_df,
        ensemble_bootstrap_df,
    ) = stratified_patient_bootstrap(
        patient_df=(
            ensemble_patient_df
        ),

        repetitions=(
            TRAINING_CONFIG[
                "bootstrap_repetitions"
            ]
        ),

        seed=(
            TRAINING_CONFIG[
                "bootstrap_seed"
            ]
            +
            300000
            +
            candidate_index
        ),
    )

    ensemble_point_estimates = {
        "qwk": (
            ensemble_patient_metrics[
                "qwk"
            ]
        ),
        "mae": (
            ensemble_patient_metrics[
                "mae"
            ]
        ),
        "macro_f1": (
            ensemble_patient_metrics[
                "macro_f1"
            ]
        ),
        "accuracy": (
            ensemble_patient_metrics[
                "accuracy"
            ]
        ),
    }

    ensemble_ci_df.insert(
        1,
        "point_estimate",
        ensemble_ci_df[
            "metric"
        ].map(
            ensemble_point_estimates
        ),
    )

    ensemble_image_path = (
        ensemble_directory
        / "oof_seed_ensemble_image_predictions.csv"
    )

    ensemble_patient_path = (
        ensemble_directory
        / "oof_seed_ensemble_patient_predictions.csv"
    )

    ensemble_ci_path = (
        ensemble_directory
        / "oof_seed_ensemble_patient_bootstrap_ci.csv"
    )

    ensemble_bootstrap_path = (
        ensemble_directory
        / "oof_seed_ensemble_patient_bootstrap_samples.csv"
    )

    atomic_csv_save(
        ensemble_image_df,
        ensemble_image_path,
        index=False,
    )

    atomic_csv_save(
        ensemble_patient_df,
        ensemble_patient_path,
        index=False,
    )

    atomic_csv_save(
        ensemble_ci_df,
        ensemble_ci_path,
        index=False,
    )

    atomic_csv_save(
        ensemble_bootstrap_df,
        ensemble_bootstrap_path,
        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            ensemble_patient_df[
                "y_true"
            ],
            ensemble_patient_df[
                "y_pred"
            ],
        ),

        ensemble_directory
        / "oof_seed_ensemble_patient_confusion_matrix.csv",

        index=True,
    )

    ensemble_ci_lookup = (
        ensemble_ci_df
        .set_index("metric")
    )

    seed_ensemble_results[
        candidate_name
    ] = {
        "candidate_name": candidate_name,

        "display_name": (
            CANDIDATES[
                candidate_name
            ]["display_name"]
        ),

        "n_seeds": int(
            len(
                ALL_CONFIRMATION_SEEDS
            )
        ),

        "oof_image_qwk": float(
            ensemble_image_metrics[
                "qwk"
            ]
        ),

        "oof_image_mae": float(
            ensemble_image_metrics[
                "mae"
            ]
        ),

        "oof_image_macro_f1": float(
            ensemble_image_metrics[
                "macro_f1"
            ]
        ),

        "oof_image_accuracy": float(
            ensemble_image_metrics[
                "accuracy"
            ]
        ),

        "oof_patient_qwk": float(
            ensemble_patient_metrics[
                "qwk"
            ]
        ),

        "oof_patient_qwk_ci_low": float(
            ensemble_ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "oof_patient_qwk_ci_high": float(
            ensemble_ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "oof_patient_mae": float(
            ensemble_patient_metrics[
                "mae"
            ]
        ),

        "oof_patient_mae_ci_low": float(
            ensemble_ci_lookup.loc[
                "mae",
                "lower_95",
            ]
        ),

        "oof_patient_mae_ci_high": float(
            ensemble_ci_lookup.loc[
                "mae",
                "upper_95",
            ]
        ),

        "oof_patient_macro_f1": float(
            ensemble_patient_metrics[
                "macro_f1"
            ]
        ),

        "oof_patient_accuracy": float(
            ensemble_patient_metrics[
                "accuracy"
            ]
        ),

        "image_predictions_path": str(
            ensemble_image_path
        ),

        "patient_predictions_path": str(
            ensemble_patient_path
        ),
    }

    seed_ensemble_patient_predictions[
        candidate_name
    ] = ensemble_patient_df


seed_ensemble_results_df = pd.DataFrame(
    list(
        seed_ensemble_results.values()
    )
)

seed_ensemble_results_path = (
    OUTPUT_ROOT
    / "vit_seed_ensemble_results.csv"
)

atomic_csv_save(
    seed_ensemble_results_df,
    seed_ensemble_results_path,
    index=False,
)


# ============================================================
# 33. Seed-ensemble paired bootstrap
# ============================================================

(
    ensemble_paired_summary_df,
    ensemble_paired_samples_df,
) = paired_patient_bootstrap(
    patient_df_a=(
        seed_ensemble_patient_predictions[
            CANDIDATE_A
        ]
    ),

    patient_df_b=(
        seed_ensemble_patient_predictions[
            CANDIDATE_B
        ]
    ),

    repetitions=(
        TRAINING_CONFIG[
            "bootstrap_repetitions"
        ]
    ),

    seed=(
        TRAINING_CONFIG[
            "bootstrap_seed"
        ]
        +
        400000
    ),

    candidate_a=CANDIDATE_A,
    candidate_b=CANDIDATE_B,
)

ensemble_paired_summary_path = (
    OUTPUT_ROOT
    / "vit_seed_ensemble_paired_bootstrap_summary.csv"
)

ensemble_paired_samples_path = (
    OUTPUT_ROOT
    / "vit_seed_ensemble_paired_bootstrap_samples.csv"
)

atomic_csv_save(
    ensemble_paired_summary_df,
    ensemble_paired_summary_path,
    index=False,
)

atomic_csv_save(
    ensemble_paired_samples_df,
    ensemble_paired_samples_path,
    index=False,
)


# ============================================================
# 34. Decision summary
# ============================================================

top_ranked_candidate = (
    multiseed_summary_ranked_df
    .iloc[0][
        "candidate_name"
    ]
)

multiseed_qwk_row = (
    paired_multiseed_summary_df[
        paired_multiseed_summary_df[
            "metric"
        ]
        ==
        "qwk"
    ]
    .iloc[0]
)

multiseed_mae_row = (
    paired_multiseed_summary_df[
        paired_multiseed_summary_df[
            "metric"
        ]
        ==
        "mae"
    ]
    .iloc[0]
)

qwk_ci_includes_zero = bool(
    multiseed_qwk_row[
        "lower_95"
    ]
    <=
    0
    <=
    multiseed_qwk_row[
        "upper_95"
    ]
)

mae_ci_includes_zero = bool(
    multiseed_mae_row[
        "lower_95"
    ]
    <=
    0
    <=
    multiseed_mae_row[
        "upper_95"
    ]
)

if qwk_ci_includes_zero:
    interpretation = (
        "The paired bootstrap confidence interval for "
        "the mean patient-level QWK difference across "
        "seeds includes zero. The two shortlisted losses "
        "are not clearly separated by QWK under this "
        "development-only confirmation protocol."
    )

else:
    qwk_difference = float(
        multiseed_qwk_row[
            "point_difference"
        ]
    )

    if qwk_difference > 0:
        interpretation = (
            "The paired bootstrap comparison favors the "
            "expected ordinal-distance loss for mean "
            "patient-level QWK across seeds."
        )

    else:
        interpretation = (
            "The paired bootstrap comparison favors the "
            "expected-value ordinal MAE loss for mean "
            "patient-level QWK across seeds."
        )

decision_summary = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "confirmation_plan_fingerprint": (
        confirmation_plan_fingerprint
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        "mean_probability"
    ),

    "candidates": list(
        CANDIDATES.keys()
    ),

    "training_seeds": [
        int(seed)
        for seed in ALL_CONFIRMATION_SEEDS
    ],

    "top_ranked_development_candidate": (
        top_ranked_candidate
    ),

    "paired_multiseed_qwk_difference_definition": (
        f"{CANDIDATE_A} - {CANDIDATE_B}"
    ),

    "paired_multiseed_qwk_point_difference": float(
        multiseed_qwk_row[
            "point_difference"
        ]
    ),

    "paired_multiseed_qwk_95_ci": [
        float(
            multiseed_qwk_row[
                "lower_95"
            ]
        ),

        float(
            multiseed_qwk_row[
                "upper_95"
            ]
        ),
    ],

    "paired_multiseed_qwk_ci_includes_zero": (
        qwk_ci_includes_zero
    ),

    "paired_multiseed_mae_point_difference": float(
        multiseed_mae_row[
            "point_difference"
        ]
    ),

    "paired_multiseed_mae_95_ci": [
        float(
            multiseed_mae_row[
                "lower_95"
            ]
        ),

        float(
            multiseed_mae_row[
                "upper_95"
            ]
        ),
    ],

    "paired_multiseed_mae_ci_includes_zero": (
        mae_ci_includes_zero
    ),

    "interpretation": (
        interpretation
    ),

    "manuscript_guidance": (
        "Use the prespecified development-only ranking "
        "to identify the representative ViT configuration. "
        "Avoid claiming superiority when the paired "
        "confidence interval includes zero."
    ),
}

decision_summary_path = (
    OUTPUT_ROOT
    / "vit_multiseed_confirmation_decision_summary.json"
)

atomic_json_save(
    decision_summary,
    decision_summary_path,
)


# ============================================================
# 35. Display final results
# ============================================================

print("\n" + "=" * 120)
print("VIT MULTI-SEED CONFIRMATION: PER-SEED RESULTS")
print("=" * 120)

display(
    per_seed_results_df[
        [
            "candidate_name",
            "training_seed",
            "source_type",

            "oof_patient_qwk",
            "oof_patient_qwk_ci_low",
            "oof_patient_qwk_ci_high",

            "oof_patient_mae",
            "oof_patient_mae_ci_low",
            "oof_patient_mae_ci_high",

            "oof_patient_macro_f1",
            "oof_patient_accuracy",

            "oof_image_qwk",
            "oof_image_mae",
        ]
    ]
)

print("\n" + "=" * 120)
print("VIT MULTI-SEED CONFIRMATION: MEAN ± SD")
print("=" * 120)

display(
    multiseed_summary_ranked_df[
        [
            "development_rank",
            "candidate_name",
            "n_training_seeds",

            "oof_patient_qwk_mean",
            "oof_patient_qwk_std",

            "oof_patient_mae_mean",
            "oof_patient_mae_std",

            "oof_patient_macro_f1_mean",
            "oof_patient_macro_f1_std",

            "oof_patient_accuracy_mean",
            "oof_patient_accuracy_std",

            "oof_image_qwk_mean",
            "oof_image_qwk_std",

            "oof_image_mae_mean",
            "oof_image_mae_std",
        ]
    ]
)

print("\n" + "=" * 120)
print("PAIRED BOOTSTRAP OF MEAN METRICS ACROSS SEEDS")
print("=" * 120)

display(
    paired_multiseed_summary_df
)

print("\n" + "=" * 120)
print("SEED-ENSEMBLE OOF RESULTS")
print("=" * 120)

display(
    seed_ensemble_results_df
)

print("\n" + "=" * 120)
print("SEED-ENSEMBLE PAIRED BOOTSTRAP")
print("=" * 120)

display(
    ensemble_paired_summary_df
)

print("\n" + "=" * 120)
print("CONFIRMATION INTERPRETATION")
print("=" * 120)

print(interpretation)

print(
    "\nTop-ranked development candidate:",
    top_ranked_candidate,
)

print(
    "\nOfficial LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)


# ============================================================
# 36. Final status
# ============================================================

print("\n" + "=" * 120)
print("02_vit_loss_multiseed_confirmation COMPLETED")
print("=" * 120)

print(
    "Output directory:",
    OUTPUT_ROOT,
)

print(
    "Per-seed results:",
    per_seed_results_path,
)

print(
    "Multi-seed summary:",
    multiseed_summary_path,
)

print(
    "Paired bootstrap by seed:",
    paired_by_seed_summary_path,
)

print(
    "Paired bootstrap across seeds:",
    paired_multiseed_summary_path,
)

print(
    "Seed-ensemble results:",
    seed_ensemble_results_path,
)

print(
    "Seed-ensemble paired bootstrap:",
    ensemble_paired_summary_path,
)

print(
    "Decision summary:",
    decision_summary_path,
)

print(
    "\nResume policy: completed folds and completed seeds "
    "are reused. An interrupted fold resumes after its "
    "last completed epoch."
)

print(
    "Only the epoch interrupted by a Colab disconnection "
    "must be repeated."
)